In [1]:
import numpy as np
import pandas as pd
import time
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import (
    precision_score, recall_score, f1_score, accuracy_score, 
    roc_auc_score, confusion_matrix
)
from sklearn.ensemble import StackingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.naive_bayes import GaussianNB
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SVMSMOTE, SMOTE, SMOTENC,ADASYN
import numpy as np
import pandas as pd
import time
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import (
    precision_score, recall_score, f1_score, accuracy_score, 
    roc_auc_score, confusion_matrix
)
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.naive_bayes import GaussianNB
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SVMSMOTE

# Data

In [2]:
df_train = pd.read_csv(r"C:\mydata\G8Vitamin\data\final\24082025\processed_train.csv")
df_test = pd.read_csv(r"C:\mydata\G8Vitamin\data\final\24082025\processed_test.csv")

In [3]:
df_train.columns

Index(['Gender', 'Age', 'Race', 'familysize', 'PIR', 'BMI',
       'WaistCircumference', 'FastingGlucose', 'ALT', 'AST',
       'AlkalinePhosphotase', 'Triglycerides', 'UricAcid', 'Creatinine',
       'HDLCholesterol', 'LDLCholesterol', 'Hemoglobin', 'Hematocrit',
       'MeanCellVolumn', 'MeanCellHemoglobin', 'RedCellDistributionWidth',
       'PlateletCount', 'MeanPlateletVolume', 'Hba1c', 'VitaminD', 'SmokeFam',
       'milk_consumption', 'YearStart', 'label'],
      dtype='object')

In [4]:
columns_remove = [
    'VitaminD',
    'YearStart',
]

In [5]:
df_train = df_train[df_train['milk_consumption']<=3]
df_test = df_test[df_test['milk_consumption']<=3]

In [6]:
df_train.drop(columns=columns_remove, inplace=True)
df_test.drop(columns=columns_remove, inplace=True)

In [7]:
category_columns = [
    'Gender','Race' ,'SmokeFam','Hba1c','label'
]

In [8]:
frequence = [
    'milk_consumption'
]

In [8]:
unuseful_features = ['SmokeFam','WaistCircumference','AST','ALT','AlkalinePhosphotase','MeanCellHemoglobin','PlateletCount', 'MeanPlateletVolume','familysize']

In [9]:
unuseful_features = ['SmokeFam','WaistCircumference','familysize','UricAcid','LDLCholesterol','Hematocrit','PlateletCount', 'MeanPlateletVolume']

In [10]:
df_train.drop(columns=unuseful_features,inplace=True)
df_test = df_test[df_train.columns]

# Training and evaluation

## stacking setup

In [11]:
#model definition
from sklearn.ensemble import AdaBoostClassifier


dt_setups = {
    "Shallow": DecisionTreeClassifier(criterion="gini", max_depth=5, min_samples_split=10, min_samples_leaf=4, random_state=42),
    "Balanced": DecisionTreeClassifier(criterion="entropy", max_depth=10, min_samples_split=5, min_samples_leaf=2, random_state=42),
    "Deep": DecisionTreeClassifier(criterion="gini", max_depth=20, min_samples_split=2, min_samples_leaf=1, random_state=42),
    "Randomized": DecisionTreeClassifier(criterion="log_loss", splitter="random", max_depth=None, min_samples_split=20, min_samples_leaf=6, random_state=42),
    "WideFeatures": DecisionTreeClassifier(criterion="entropy", splitter="best", max_depth=30, min_samples_split=5, min_samples_leaf=2, max_features="sqrt", random_state=42)
}
gbc_setups = {
    "Balanced": GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42),
    "ShallowFast": GradientBoostingClassifier(n_estimators=80, learning_rate=0.2, max_depth=2, subsample=0.8, random_state=42),
    "Regularized": GradientBoostingClassifier(n_estimators=200, learning_rate=0.01, max_depth=4, min_samples_split=20, min_samples_leaf=5, random_state=42),
    "Deep": GradientBoostingClassifier(n_estimators=300, learning_rate=0.05, max_depth=6, subsample=0.9, random_state=42),
    "RobustSubsample": GradientBoostingClassifier(n_estimators=150, learning_rate=0.08, max_depth=5, subsample=0.7, max_features="sqrt", random_state=42),
    "Conservative": GradientBoostingClassifier(n_estimators=500, learning_rate=0.01, max_depth=3, subsample=0.8, max_features="log2", random_state=42)
}

rf_setups = {
    "Balanced": RandomForestClassifier(
        n_estimators=100, max_depth=None, random_state=42
    ),
    "ShallowFast": RandomForestClassifier(
        n_estimators=50, max_depth=5, max_features="sqrt", random_state=42
    ),
    "Deep": RandomForestClassifier(
        n_estimators=300, max_depth=20, min_samples_split=5,
        min_samples_leaf=2, random_state=42
    ),
    "Regularized": RandomForestClassifier(
        n_estimators=100, max_depth=10, min_samples_split=10,
        min_samples_leaf=4, max_features=0.6, random_state=42
    ),
    "Conservative": RandomForestClassifier(
        n_estimators=500, max_depth=None, min_samples_split=20,
        min_samples_leaf=10, max_features="log2", random_state=42
    ),
    "RobustSubsample": RandomForestClassifier(#no ok
        n_estimators=150, max_depth=15, min_samples_split=5,
        min_samples_leaf=3, bootstrap=True, max_features="sqrt",
        random_state=42
    ),
    "Lightweight": RandomForestClassifier(
        n_estimators=50, max_depth=8, max_features=0.5, random_state=42
    ),
    "Heavy": RandomForestClassifier(
        n_estimators=1000, max_depth=None, min_samples_split=2,
        min_samples_leaf=1, max_features=None, random_state=42, n_jobs=-1
    ),
}

#Best: GradientBoostingClassifier(n_estimators=80, learning_rate=0.05, random_state=42)
base_learners = [
    ('lightgbm', LGBMClassifier(n_estimators=100, max_depth=6, num_leaves=31, learning_rate=0.05, random_state=42)),
    # ('xgboost', XGBClassifier(n_estimators=80, learning_rate=0.066, random_state=42, verbosity=0)),
    # ('gb', GradientBoostingClassifier(n_estimators=80, learning_rate=0.05, random_state=42)),
    ('RandomForest', rf_setups['Regularized']),
    # ('dt', dt_setups['Shallow']),  
    # ('LogisticRegression', LogisticRegression(max_iter=1000, random_state=42)),
    #('SVM', SVC(kernel='rbf', C=10, gamma=0.01, class_weight='balanced', probability=True, random_state=42)),
    ('adaboost',AdaBoostClassifier(n_estimators=300,learning_rate=0.9, random_state=0),),
    ('nb', GaussianNB(var_smoothing= 1e-10)),
]


# ====== 3) Meta-learner ======
# meta_learner = LogisticRegression(max_iter=1000, random_state=42)
meta_learner = LogisticRegression(
    max_iter=3000,
    solver='saga',          # saga supports L1, L2, elasticnet
    # penalty='elasticnet',   # mix between L1 and L2
    C=0.2,                  # regularization strength
    # class_weight='balanced',
    random_state=42
)
# ====== 4) Stacking classifier ======
stacking_clf = StackingClassifier(
    estimators=base_learners,
    final_estimator=meta_learner,
    stack_method="predict_proba",  # use probabilities from base models
    n_jobs=-1
)

## All classifier setup

In [13]:

from sklearn.ensemble import AdaBoostClassifier


classifiers = {
    'LightGBM': LGBMClassifier(n_estimators=120, max_depth=6, num_leaves=31, learning_rate=0.05, random_state=42),
    'XGBoost': XGBClassifier(n_estimators=80, learning_rate=0.066, random_state=42, verbosity=0),
    'GradiantBoosting': GradientBoostingClassifier(n_estimators=80, learning_rate=0.05, random_state=42),
    'RandomForest': RandomForestClassifier(n_estimators=100, random_state=42),
    'GradientBoosting': GradientBoostingClassifier(n_estimators=100, learning_rate=0.05, random_state=42),
    'Naive Bayes': GaussianNB(var_smoothing=1e-10),
    # 'SVM': SVC(kernel='rbf', C=10, gamma=0.01, class_weight='balanced', probability=True, random_state=42),
    'Stacking':stacking_clf,
}

In [16]:
df_train['milk_consumption'].value_counts()

milk_consumption
3.0    7582
2.0    4274
0.0    2180
1.0    2173
Name: count, dtype: int64

In [14]:
from sklearn.preprocessing import OrdinalEncoder
# ====== 0) Train/test split ======
X_train_raw = df_train.drop(columns='label')
y_train = df_train['label']
X_test_raw = df_test.drop(columns=['label'])
y_test = df_test['label']

categorical_cols = ['Gender','Race','Hba1c','milk_consumption']
print(categorical_cols)
numeric_cols = [col for col in df_train.columns if col not in categorical_cols and col !='label']
print(numeric_cols)
categorical_cols = ['Gender','Race']
print(categorical_cols)
# ====== 1) Preprocessor ======
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore', drop='first'), categorical_cols),
        ('num', StandardScaler(), numeric_cols),
    ],
    remainder='passthrough'
)

# ====== 3) SVMSMOTE setup ======
svmsmote = SVMSMOTE(
    random_state=42,
    sampling_strategy='auto',
    k_neighbors=20,
    m_neighbors=40,
    svm_estimator=SVC(kernel="rbf", gamma='auto', C=1)
)

# ====== 4) Wrapper for imblearn pipelines ======
class ImblearnWrapper(BaseEstimator, ClassifierMixin):
    def __init__(self, pipeline):
        self.pipeline = pipeline
    def fit(self, X, y):
        self.pipeline.fit(X, y)
        return self
    def predict(self, X):
        return self.pipeline.predict(X)
    def predict_proba(self, X):
        return self.pipeline.predict_proba(X)


# ====== 5) Train & evaluate ======
rows = []

for name, clf in classifiers.items():
    print(f"\n🚀 Training {name} with SVMSMOTE...")
    pipeline = ImbPipeline(steps=[
        ('preprocessor', preprocessor),
        ('smote', svmsmote),
        ('classifier', clf)
    ])
    wrapped_model = ImblearnWrapper(pipeline)

    try:
        start = time.time()
        wrapped_model.fit(X_train_raw, y_train)
        train_time = time.time() - start

        y_pred = wrapped_model.predict(X_test_raw)
        y_proba = wrapped_model.predict_proba(X_test_raw)

        # ===== Global metrics =====
        acc = accuracy_score(y_test, y_pred)
        f1_macro = f1_score(y_test, y_pred, average='macro', zero_division=0)
        f1_weighted = f1_score(y_test, y_pred, average='weighted', zero_division=0)

        # AUC
        if len(np.unique(y_test)) == 2:
            auc = roc_auc_score(y_test, y_proba[:, 1])
        else:
            auc = roc_auc_score(y_test, y_proba, multi_class='ovr', average='macro')

        # ===== Per-class metrics =====
        cm = confusion_matrix(y_test, y_pred, labels=[0,1])
        classes = [0,1]
        for i, cls in enumerate(classes):
            TP = cm[i,i]
            FN = cm[i,:].sum() - TP
            FP = cm[:,i].sum() - TP
            TN = cm.sum() - (TP+FN+FP)

            PPV = TP/(TP+FP) if (TP+FP)>0 else 0  # Precision
            NPV = TN/(TN+FN) if (TN+FN)>0 else 0
            SEN = TP/(TP+FN) if (TP+FN)>0 else 0  # Recall
            SPE = TN/(TN+FP) if (TN+FP)>0 else 0

            # Add row with per-class metrics
            rows.append({
                "Model": name,
                "Label": cls,
                "Training time": round(train_time,4) if cls==0 else None,
                "ACC": round(acc,4) if cls==0 else None,
                "F1_macro": round(f1_macro,4) if cls==0 else None,
                "F1_weighted": round(f1_weighted,4) if cls==0 else None,
                "AUC": round(auc,4) if cls==0 else None,
                "PPV": round(PPV,4),
                "NPV": round(NPV,4),
                "SEN": round(SEN,4),
                "SPE": round(SPE,4),
            })

        print(f"✅ {name} - ACC={acc:.4f}, F1_macro={f1_macro:.4f}, AUC={auc:.4f}")

    except Exception as e:
        print(f"❌ Error training {name}: {e}")


# ====== 6) Save results ======
results_df = pd.DataFrame(rows)

print("\n📊 FINAL TABLE (Excel format-ready):")
print(results_df)

results_df.to_csv("deletenull_keep_alt6.csv", index=False)
print("\n✅ Exported to expected_output_table.csv")


['Gender', 'Race', 'Hba1c', 'milk_consumption']
['Age', 'PIR', 'BMI', 'FastingGlucose', 'ALT', 'AST', 'AlkalinePhosphotase', 'Triglycerides', 'Creatinine', 'HDLCholesterol', 'Hemoglobin', 'MeanCellVolumn', 'MeanCellHemoglobin', 'RedCellDistributionWidth']
['Gender', 'Race']

🚀 Training LightGBM with SVMSMOTE...
[LightGBM] [Info] Number of positive: 12380, number of negative: 12380
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001043 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 5351
[LightGBM] [Info] Number of data points in the train set: 24760, number of used features: 21
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No

c:\mydata\G8Vitamin\.venv\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\mydata\G8Vitamin\.venv\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


✅ XGBoost - ACC=0.6996, F1_macro=0.6638, AUC=0.7419

🚀 Training GradiantBoosting with SVMSMOTE...
✅ GradiantBoosting - ACC=0.6862, F1_macro=0.6595, AUC=0.7378

🚀 Training RandomForest with SVMSMOTE...
✅ RandomForest - ACC=0.7199, F1_macro=0.6568, AUC=0.7297

🚀 Training GradientBoosting with SVMSMOTE...
✅ GradientBoosting - ACC=0.6907, F1_macro=0.6637, AUC=0.7413

🚀 Training Naive Bayes with SVMSMOTE...
✅ Naive Bayes - ACC=0.6274, F1_macro=0.6100, AUC=0.6771

🚀 Training Stacking with SVMSMOTE...


c:\mydata\G8Vitamin\.venv\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\mydata\G8Vitamin\.venv\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


✅ Stacking - ACC=0.7233, F1_macro=0.6781, AUC=0.7393

📊 FINAL TABLE (Excel format-ready):
               Model  Label  Training time     ACC  F1_macro  F1_weighted  \
0           LightGBM      0         5.4824  0.7132    0.6700       0.7153   
1           LightGBM      1            NaN     NaN       NaN          NaN   
2            XGBoost      0         3.9182  0.6996    0.6638       0.7054   
3            XGBoost      1            NaN     NaN       NaN          NaN   
4   GradiantBoosting      0         8.8709  0.6862    0.6595       0.6956   
5   GradiantBoosting      1            NaN     NaN       NaN          NaN   
6       RandomForest      0         8.0195  0.7199    0.6568       0.7126   
7       RandomForest      1            NaN     NaN       NaN          NaN   
8   GradientBoosting      0        19.0136  0.6907    0.6637       0.6999   
9   GradientBoosting      1            NaN     NaN       NaN          NaN   
10       Naive Bayes      0         3.6573  0.6274    0.6100   

In [14]:
import time
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score,
    confusion_matrix, roc_curve, precision_recall_curve, auc
)
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SVMSMOTE
from sklearn.svm import SVC

# ====== 0) Train/test split ======
X_train_raw = df_train.drop(columns='label')
y_train = df_train['label']
X_test_raw = df_test.drop(columns=['label'])
y_test = df_test['label']

categorical_cols = ['Gender', 'Race']
numeric_cols = [col for col in df_train.columns if col not in categorical_cols and col != 'label']

# ====== 1) Preprocessor ======
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore', drop='first'), categorical_cols),
        ('num', StandardScaler(), numeric_cols),
    ],
    remainder='passthrough'
)

# ====== 2) Classifiers ======
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.naive_bayes import GaussianNB
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

# classifiers = {
#     "LightGBM": LGBMClassifier(random_state=42),
#     "XGBoost": XGBClassifier(eval_metric="logloss", use_label_encoder=False, random_state=42),
#     "RandomForest": RandomForestClassifier(random_state=42),
#     "GradientBoosting": GradientBoostingClassifier(random_state=42),
#     "Naive Bayes": GaussianNB(),
#     "SVM": SVC(probability=True, random_state=42),
# }

# ====== 3) SVMSMOTE setup ======
svmsmote = SVMSMOTE(
    random_state=42,
    sampling_strategy='auto',
    k_neighbors=20,
    m_neighbors=40,
    svm_estimator=SVC(kernel="rbf", gamma='auto', C=1)
)

# ====== 4) Wrapper for imblearn pipelines ======
class ImblearnWrapper(BaseEstimator, ClassifierMixin):
    def __init__(self, pipeline):
        self.pipeline = pipeline
    def fit(self, X, y):
        self.pipeline.fit(X, y)
        return self
    def predict(self, X):
        return self.pipeline.predict(X)
    def predict_proba(self, X):
        return self.pipeline.predict_proba(X)

# ====== 5) Train & evaluate ======
rows_detailed = []
rows_summary = []

data_versions = [
    "Full_data_process (50%) + SVMSMOTE",
    # "Full_data_process (70%) + SVMSMOTE",
    # "Full_data_process (100%) + SVMSMOTE"
]

for version in data_versions:
    summary_row = {
        "Data versioning": version,
        "Completeness": "✔",   # fill with real %
        "Consistency": "✔",    # fill with real %
    }

    for name, clf in classifiers.items():
        print(f"\n🚀 Training {name} on {version}...")

        pipeline = ImbPipeline(steps=[
            ('preprocessor', preprocessor),
            ('smote', svmsmote),
            ('classifier', clf)
        ])
        wrapped_model = ImblearnWrapper(pipeline)

        try:
            start = time.time()
            wrapped_model.fit(X_train_raw, y_train)
            train_time = time.time() - start

            y_pred = wrapped_model.predict(X_test_raw)
            y_proba = wrapped_model.predict_proba(X_test_raw)

            # Global metrics
            acc = accuracy_score(y_test, y_pred)
            f1_macro = f1_score(y_test, y_pred, average='macro', zero_division=0)
            f1_weighted = f1_score(y_test, y_pred, average='weighted', zero_division=0)

            if len(np.unique(y_test)) == 2:
                auc_roc = roc_auc_score(y_test, y_proba[:, 1])
                auc_pr = auc(*precision_recall_curve(y_test, y_proba[:, 1])[1::-1])
            else:
                auc_roc = roc_auc_score(y_test, y_proba, multi_class='ovr', average='macro')
                auc_pr = np.nan

            # Per-class metrics
            cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
            for i, cls in enumerate([0, 1]):
                TP = cm[i, i]
                FN = cm[i, :].sum() - TP
                FP = cm[:, i].sum() - TP
                TN = cm.sum() - (TP + FN + FP)

                PPV = TP/(TP+FP) if (TP+FP) > 0 else 0
                NPV = TN/(TN+FN) if (TN+FN) > 0 else 0
                SEN = TP/(TP+FN) if (TP+FN) > 0 else 0
                SPE = TN/(TN+FP) if (TN+FP) > 0 else 0

                rows_detailed.append({
                    "Data versioning": version,
                    "Model": name,
                    "Label": cls,
                    "Training time": round(train_time, 4) if cls == 0 else None,
                    "ACC": round(acc, 4) if cls == 0 else None,
                    "F1_macro": round(f1_macro, 4) if cls == 0 else None,
                    "F1_weighted": round(f1_weighted, 4) if cls == 0 else None,
                    "ROC_AUC": round(auc_roc, 4) if cls == 0 else None,
                    "PR_AUC": round(auc_pr, 4) if cls == 0 else None,
                    "TP": TP, "FP": FP, "FN": FN, "TN": TN,
                    "PPV": round(PPV, 4), "NPV": round(NPV, 4),
                    "SEN": round(SEN, 4), "SPE": round(SPE, 4),
                })

                # Summary table F1 per label
                summary_row[f"F1_label{cls}_{name}"] = round(f1_score(y_test, y_pred, pos_label=cls), 3)

            # Summary row
            summary_row["Training time(seconds)"] = round(train_time, 3)
            summary_row[name] = round(acc, 3)

            print(f"✅ {name} - ACC={acc:.4f}, F1_macro={f1_macro:.4f}, ROC_AUC={auc_roc:.4f}, PR_AUC={auc_pr:.4f}")

        except Exception as e:
            print(f"❌ Error training {name}: {e}")

    rows_summary.append(summary_row)

# ====== 6) Save results ======
detailed_df = pd.DataFrame(rows_detailed)
summary_df = pd.DataFrame(rows_summary)

# Order summary columns like your Excel template
summary_cols = (
    ["Data versioning", "Completeness", "Consistency", "Training time(seconds)"] +
    list(classifiers.keys()) +
    [f"F1_label0_{m}" for m in classifiers.keys()] +
    [f"F1_label1_{m}" for m in classifiers.keys()]
)
summary_df = summary_df[summary_cols]

print("\n📊 SUMMARY TABLE (Excel format-ready):")
print(summary_df.head())

detailed_df.to_csv("metrics_detailed.csv", index=False)
summary_df.to_excel("metrics_summary.xlsx", index=False)

print("\n✅ Exported detailed metrics to metrics_detailed.csv")
print("✅ Exported summary results to metrics_summary.xlsx")



🚀 Training LightGBM on Full_data_process (50%) + SVMSMOTE...
[LightGBM] [Info] Number of positive: 12380, number of negative: 12380
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000851 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 5098
[LightGBM] [Info] Number of data points in the train set: 24760, number of used features: 21
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with posit

c:\mydata\G8Vitamin\.venv\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\mydata\G8Vitamin\.venv\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


✅ XGBoost - ACC=0.7048, F1_macro=0.6672, ROC_AUC=0.7406, PR_AUC=0.5524

🚀 Training GradiantBoosting on Full_data_process (50%) + SVMSMOTE...
✅ GradiantBoosting - ACC=0.6806, F1_macro=0.6537, ROC_AUC=0.7366, PR_AUC=0.5473

🚀 Training RandomForest on Full_data_process (50%) + SVMSMOTE...
✅ RandomForest - ACC=0.7225, F1_macro=0.6560, ROC_AUC=0.7366, PR_AUC=0.5436

🚀 Training GradientBoosting on Full_data_process (50%) + SVMSMOTE...
✅ GradientBoosting - ACC=0.6827, F1_macro=0.6544, ROC_AUC=0.7380, PR_AUC=0.5494

🚀 Training Naive Bayes on Full_data_process (50%) + SVMSMOTE...
✅ Naive Bayes - ACC=0.6168, F1_macro=0.6002, ROC_AUC=0.6681, PR_AUC=0.4582

📊 SUMMARY TABLE (Excel format-ready):
                      Data versioning Completeness Consistency  \
0  Full_data_process (50%) + SVMSMOTE            ✔           ✔   

   Training time(seconds)  LightGBM  XGBoost  GradiantBoosting  RandomForest  \
0                   4.009      0.71    0.705             0.681         0.722   

   GradientBoo

In [13]:
import time
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score,
    confusion_matrix, precision_recall_curve, auc
)
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SVMSMOTE
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.naive_bayes import GaussianNB
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

# ====== 0) Train/test split ======
X_train_raw = df_train.drop(columns='label')
y_train = df_train['label']
X_test_raw = df_test.drop(columns=['label'])
y_test = df_test['label']

categorical_cols = ['Gender', 'Race']
numeric_cols = [col for col in df_train.columns if col not in categorical_cols and col != 'label']

# ====== 1) Preprocessor ======
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore', drop='first'), categorical_cols),
        ('num', StandardScaler(), numeric_cols),
    ],
    remainder='passthrough'
)

# ====== 2) Classifiers ======
# classifiers = {
#     "LightGBM": LGBMClassifier(random_state=42),
#     "XGBoost": XGBClassifier(eval_metric="logloss", use_label_encoder=False, random_state=42),
#     "RandomForest": RandomForestClassifier(random_state=42),
#     "GradientBoosting": GradientBoostingClassifier(random_state=42),
#     "Naive Bayes": GaussianNB(),
#     "SVM": SVC(probability=True, random_state=42),
# }

# ====== 3) SVMSMOTE setup ======
svmsmote = SVMSMOTE(
    random_state=42,
    sampling_strategy='auto',
    k_neighbors=20,
    m_neighbors=40,
    svm_estimator=SVC(kernel="rbf", gamma='auto', C=1)
)

# ====== 4) Wrapper for imblearn pipelines ======
class ImblearnWrapper(BaseEstimator, ClassifierMixin):
    def __init__(self, pipeline):
        self.pipeline = pipeline
    def fit(self, X, y):
        self.pipeline.fit(X, y)
        return self
    def predict(self, X):
        return self.pipeline.predict(X)
    def predict_proba(self, X):
        return self.pipeline.predict_proba(X)

# ====== 5) Train & evaluate ======
rows_detailed = []
rows_summary = []

data_versions = [
    "Full_data_process (50%) + SVMSMOTE",
    "Full_data_process (70%) + SVMSMOTE",
    "Full_data_process (100%) + SVMSMOTE"
]

for version in data_versions:
    summary_row = {
        "Data versioning": version,
        "Completeness": "✔",
        "Consistency": "✔",
    }

    for name, clf in classifiers.items():
        print(f"\n🚀 Training {name} on {version}...")

        pipeline = ImbPipeline(steps=[
            ('preprocessor', preprocessor),
            ('smote', svmsmote),
            ('classifier', clf)
        ])
        wrapped_model = ImblearnWrapper(pipeline)

        try:
            start = time.time()
            wrapped_model.fit(X_train_raw, y_train)
            train_time = time.time() - start

            y_pred = wrapped_model.predict(X_test_raw)
            y_proba = wrapped_model.predict_proba(X_test_raw)

            # Global metrics
            acc = accuracy_score(y_test, y_pred)
            f1_macro = f1_score(y_test, y_pred, average='macro', zero_division=0)
            f1_weighted = f1_score(y_test, y_pred, average='weighted', zero_division=0)

            if len(np.unique(y_test)) == 2:
                auc_roc = roc_auc_score(y_test, y_proba[:, 1])
                auc_pr = auc(*precision_recall_curve(y_test, y_proba[:, 1])[1::-1])
            else:
                auc_roc = roc_auc_score(y_test, y_proba, multi_class='ovr', average='macro')
                auc_pr = np.nan

            # Per-class metrics
            cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
            for i, cls in enumerate([0, 1]):
                TP = cm[i, i]
                FN = cm[i, :].sum() - TP
                FP = cm[:, i].sum() - TP
                TN = cm.sum() - (TP + FN + FP)

                precision_cls = TP/(TP+FP) if (TP+FP) > 0 else 0
                recall_cls = TP/(TP+FN) if (TP+FN) > 0 else 0
                f1_cls = 2*precision_cls*recall_cls/(precision_cls+recall_cls) if (precision_cls+recall_cls) > 0 else 0
                NPV = TN/(TN+FN) if (TN+FN) > 0 else 0
                SPE = TN/(TN+FP) if (TN+FP) > 0 else 0

                rows_detailed.append({
                    "Data versioning": version,
                    "Model": name,
                    "Label": cls,
                    "Training time": round(train_time, 4) if cls == 0 else None,
                    "ACC": round(acc, 4) if cls == 0 else None,
                    "F1_macro": round(f1_macro, 4) if cls == 0 else None,
                    "F1_weighted": round(f1_weighted, 4) if cls == 0 else None,
                    "ROC_AUC": round(auc_roc, 4) if cls == 0 else None,
                    "PR_AUC": round(auc_pr, 4) if cls == 0 else None,
                    "TP": TP, "FP": FP, "FN": FN, "TN": TN,
                    "Precision": round(precision_cls, 4),
                    "Recall": round(recall_cls, 4),
                    "F1_per_class": round(f1_cls, 4),
                    "NPV": round(NPV, 4),
                    "SPE": round(SPE, 4),
                })

                # Summary table F1 per label
                summary_row[f"F1_label{cls}_{name}"] = round(f1_score(y_test, y_pred, pos_label=cls), 3)

            # Summary row
            summary_row["Training time(seconds)"] = round(train_time, 3)
            summary_row[name] = round(acc, 3)

            print(f"✅ {name} - ACC={acc:.4f}, F1_macro={f1_macro:.4f}, ROC_AUC={auc_roc:.4f}, PR_AUC={auc_pr:.4f}")

        except Exception as e:
            print(f"❌ Error training {name}: {e}")

    rows_summary.append(summary_row)

# ====== 6) Save results ======
detailed_df = pd.DataFrame(rows_detailed)
summary_df = pd.DataFrame(rows_summary)

summary_cols = (
    ["Data versioning", "Completeness", "Consistency", "Training time(seconds)"] +
    list(classifiers.keys()) +
    [f"F1_label0_{m}" for m in classifiers.keys()] +
    [f"F1_label1_{m}" for m in classifiers.keys()]
)
summary_df = summary_df[summary_cols]

print("\n📊 SUMMARY TABLE (Excel format-ready):")
print(summary_df.head())

detailed_df.to_csv("metrics_detailed.csv", index=False)
summary_df.to_csv("metrics_summary.csv", index=False)   # switched to CSV, no openpyxl needed

print("\n✅ Exported detailed metrics to metrics_detailed.csv")
print("✅ Exported summary results to metrics_summary.csv")



🚀 Training LightGBM on Full_data_process (50%) + SVMSMOTE...
[LightGBM] [Info] Number of positive: 12380, number of negative: 12380
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001069 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 5098
[LightGBM] [Info] Number of data points in the train set: 24760, number of used features: 21
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with posit

c:\mydata\G8Vitamin\.venv\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\mydata\G8Vitamin\.venv\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


✅ XGBoost - ACC=0.7048, F1_macro=0.6672, ROC_AUC=0.7406, PR_AUC=0.5524

🚀 Training GradiantBoosting on Full_data_process (50%) + SVMSMOTE...
✅ GradiantBoosting - ACC=0.6806, F1_macro=0.6537, ROC_AUC=0.7366, PR_AUC=0.5473

🚀 Training RandomForest on Full_data_process (50%) + SVMSMOTE...
✅ RandomForest - ACC=0.7225, F1_macro=0.6560, ROC_AUC=0.7366, PR_AUC=0.5436

🚀 Training GradientBoosting on Full_data_process (50%) + SVMSMOTE...
✅ GradientBoosting - ACC=0.6827, F1_macro=0.6544, ROC_AUC=0.7380, PR_AUC=0.5494

🚀 Training Naive Bayes on Full_data_process (50%) + SVMSMOTE...
✅ Naive Bayes - ACC=0.6168, F1_macro=0.6002, ROC_AUC=0.6681, PR_AUC=0.4582

🚀 Training LightGBM on Full_data_process (70%) + SVMSMOTE...
[LightGBM] [Info] Number of positive: 12380, number of negative: 12380
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000851 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 5098
[LightGBM] [Info] 

c:\mydata\G8Vitamin\.venv\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\mydata\G8Vitamin\.venv\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


✅ XGBoost - ACC=0.7048, F1_macro=0.6672, ROC_AUC=0.7406, PR_AUC=0.5524

🚀 Training GradiantBoosting on Full_data_process (70%) + SVMSMOTE...
✅ GradiantBoosting - ACC=0.6806, F1_macro=0.6537, ROC_AUC=0.7366, PR_AUC=0.5473

🚀 Training RandomForest on Full_data_process (70%) + SVMSMOTE...
✅ RandomForest - ACC=0.7225, F1_macro=0.6560, ROC_AUC=0.7366, PR_AUC=0.5436

🚀 Training GradientBoosting on Full_data_process (70%) + SVMSMOTE...
✅ GradientBoosting - ACC=0.6827, F1_macro=0.6544, ROC_AUC=0.7380, PR_AUC=0.5494

🚀 Training Naive Bayes on Full_data_process (70%) + SVMSMOTE...
✅ Naive Bayes - ACC=0.6168, F1_macro=0.6002, ROC_AUC=0.6681, PR_AUC=0.4582

🚀 Training LightGBM on Full_data_process (100%) + SVMSMOTE...
[LightGBM] [Info] Number of positive: 12380, number of negative: 12380
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001682 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 5098
[LightGBM] [Info]

c:\mydata\G8Vitamin\.venv\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\mydata\G8Vitamin\.venv\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


✅ LightGBM - ACC=0.7095, F1_macro=0.6695, ROC_AUC=0.7427, PR_AUC=0.5575

🚀 Training XGBoost on Full_data_process (100%) + SVMSMOTE...
✅ XGBoost - ACC=0.7048, F1_macro=0.6672, ROC_AUC=0.7406, PR_AUC=0.5524

🚀 Training GradiantBoosting on Full_data_process (100%) + SVMSMOTE...
✅ GradiantBoosting - ACC=0.6806, F1_macro=0.6537, ROC_AUC=0.7366, PR_AUC=0.5473

🚀 Training RandomForest on Full_data_process (100%) + SVMSMOTE...
✅ RandomForest - ACC=0.7225, F1_macro=0.6560, ROC_AUC=0.7366, PR_AUC=0.5436

🚀 Training GradientBoosting on Full_data_process (100%) + SVMSMOTE...
✅ GradientBoosting - ACC=0.6827, F1_macro=0.6544, ROC_AUC=0.7380, PR_AUC=0.5494

🚀 Training Naive Bayes on Full_data_process (100%) + SVMSMOTE...
✅ Naive Bayes - ACC=0.6168, F1_macro=0.6002, ROC_AUC=0.6681, PR_AUC=0.4582

📊 SUMMARY TABLE (Excel format-ready):
                       Data versioning Completeness Consistency  \
0   Full_data_process (50%) + SVMSMOTE            ✔           ✔   
1   Full_data_process (70%) + SVMSMO

In [21]:
summary_df.to_excel("metrics_summary.xlsx", index=False)

## Training and evaluation

In [15]:
# --- IMPORTS ---
import pandas as pd
import numpy as np
import networkx as nx
from node2vec import Node2Vec
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics.pairwise import euclidean_distances
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from imblearn.over_sampling import SMOTE

# ====== 0) Predefined train/test ======
X_train_raw = df_train.drop(columns='label')
y_train = df_train['label']
X_test_raw = df_test.drop(columns=['label'])
y_test = df_test['label']

categorical_cols = ["Gender", "Race", "SmokeFam", "milk_consumption", "YearStart"]
numeric_cols = [col for col in X_train_raw.columns if col not in categorical_cols]

# ====== 1) Preprocess ======
preprocessor = ColumnTransformer([
    ("num", StandardScaler(), numeric_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols)
])

X_train_proc = preprocessor.fit_transform(X_train_raw)
X_test_proc = preprocessor.transform(X_test_raw)

# ====== 2) Build graph on ORIGINAL train (before SMOTE) ======
n_train = X_train_proc.shape[0]
dist_matrix = euclidean_distances(X_train_proc)
threshold = np.percentile(dist_matrix, 10)  # connect ~10% closest

G = nx.Graph()
for i in range(n_train):
    G.add_node(i)

for i in range(n_train):
    for j in range(i+1, n_train):
        if dist_matrix[i, j] < threshold:
            G.add_edge(i, j, weight=1.0/(1e-5 + dist_matrix[i, j]))

# ====== 3) Graph features for train ======
deg_cent = nx.degree_centrality(G)
bet_cent = nx.betweenness_centrality(G)
clust = nx.clustering(G)

graph_feats_train = pd.DataFrame({
    "id": list(G.nodes()),
    "deg_cent": [deg_cent[i] for i in G.nodes()],
    "bet_cent": [bet_cent[i] for i in G.nodes()],
    "clust": [clust[i] for i in G.nodes()]
})

# Node2Vec embeddings
node2vec = Node2Vec(G, dimensions=8, walk_length=10, num_walks=50, workers=1, seed=42)
model = node2vec.fit(window=5, min_count=1, batch_words=4)
embeddings = {int(node): model.wv[str(node)] for node in G.nodes()}
embeddings_df = pd.DataFrame.from_dict(embeddings, orient="index")
embeddings_df.reset_index(inplace=True)
embeddings_df.rename(columns={"index":"id"}, inplace=True)

graph_feats_train = graph_feats_train.merge(embeddings_df, on="id")
graph_feats_train.set_index("id", inplace=True)

# ====== 4) Append graph features to train/test (before SMOTE) ======
X_train_with_graph = np.hstack([X_train_proc.toarray(), graph_feats_train.values])
X_test_with_graph = np.hstack([
    X_test_proc.toarray(),
    np.zeros((X_test_proc.shape[0], graph_feats_train.shape[1]))  # test not in train graph
])

# ====== 5) Apply SMOTE on train with graph ======
smote = SMOTE(random_state=42)
X_train_bal, y_train_bal = smote.fit_resample(X_train_with_graph, y_train)

# ====== 6) Train classifier ======
clf = RandomForestClassifier(random_state=42)
clf.fit(X_train_bal, y_train_bal)
y_pred = clf.predict(X_test_with_graph)

print(classification_report(y_test, y_pred))


ModuleNotFoundError: No module named 'node2vec'

In [16]:
# --- IMPORTS ---
import pandas as pd
import numpy as np
import networkx as nx
from gensim.models import Word2Vec
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics.pairwise import euclidean_distances
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from imblearn.over_sampling import SMOTE
import random

# ====== 0) Predefined train/test ======
X_train_raw = df_train.drop(columns='label')
y_train = df_train['label']
X_test_raw = df_test.drop(columns=['label'])
y_test = df_test['label']

categorical_cols = ["Gender", "Race", "SmokeFam", "milk_consumption", "YearStart"]
numeric_cols = [col for col in X_train_raw.columns if col not in categorical_cols]

# ====== 1) Preprocess ======
preprocessor = ColumnTransformer([
    ("num", StandardScaler(), numeric_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols)
])

X_train_proc = preprocessor.fit_transform(X_train_raw)
X_test_proc = preprocessor.transform(X_test_raw)

# ====== 2) Build graph on ORIGINAL train (before SMOTE) ======
n_train = X_train_proc.shape[0]
dist_matrix = euclidean_distances(X_train_proc)
threshold = np.percentile(dist_matrix, 10)  # connect ~10% closest

G = nx.Graph()
for i in range(n_train):
    G.add_node(i)

for i in range(n_train):
    for j in range(i+1, n_train):
        if dist_matrix[i, j] < threshold:
            G.add_edge(i, j, weight=1.0/(1e-5 + dist_matrix[i, j]))

# ====== 3) Graph features for train ======
deg_cent = nx.degree_centrality(G)
bet_cent = nx.betweenness_centrality(G)
clust = nx.clustering(G)

graph_feats_train = pd.DataFrame({
    "id": list(G.nodes()),
    "deg_cent": [deg_cent[i] for i in G.nodes()],
    "bet_cent": [bet_cent[i] for i in G.nodes()],
    "clust": [clust[i] for i in G.nodes()]
})

# ====== 3b) Node2Vec replacement using random walks + Word2Vec ======
def generate_walks(G, num_walks=50, walk_length=10):
    walks = []
    nodes = list(G.nodes())
    for _ in range(num_walks):
        random.shuffle(nodes)
        for node in nodes:
            walk = [str(node)]
            current = node
            for _ in range(walk_length - 1):
                neighbors = list(G.neighbors(current))
                if neighbors:
                    current = random.choice(neighbors)
                    walk.append(str(current))
                else:
                    break
            walks.append(walk)
    return walks

walks = generate_walks(G, num_walks=50, walk_length=10)
w2v_model = Word2Vec(sentences=walks, vector_size=8, window=5, min_count=1, sg=1, workers=1, seed=42)

embeddings = {int(node): w2v_model.wv[str(node)] for node in G.nodes()}
embeddings_df = pd.DataFrame.from_dict(embeddings, orient="index")
embeddings_df.reset_index(inplace=True)
embeddings_df.rename(columns={"index":"id"}, inplace=True)

graph_feats_train = graph_feats_train.merge(embeddings_df, on="id")
graph_feats_train.set_index("id", inplace=True)

# ====== 4) Append graph features to train/test (before SMOTE) ======
X_train_with_graph = np.hstack([X_train_proc.toarray(), graph_feats_train.values])
X_test_with_graph = np.hstack([
    X_test_proc.toarray(),
    np.zeros((X_test_proc.shape[0], graph_feats_train.shape[1]))  # test not in train graph
])

# ====== 5) Apply SMOTE on train with graph ======
smote = SMOTE(random_state=42)
X_train_bal, y_train_bal = smote.fit_resample(X_train_with_graph, y_train)

# ====== 6) Train classifier ======
clf = RandomForestClassifier(random_state=42)
clf.fit(X_train_bal, y_train_bal)
y_pred = clf.predict(X_test_with_graph)

print(classification_report(y_test, y_pred))


ModuleNotFoundError: No module named 'gensim'

In [19]:
# --- IMPORTS ---
import pandas as pd
import numpy as np
import networkx as nx
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics.pairwise import euclidean_distances
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from imblearn.over_sampling import SMOTE
import random

# ====== 0) Predefined train/test ======
X_train_raw = df_train.drop(columns='label')
y_train = df_train['label']
X_test_raw = df_test.drop(columns=['label'])
y_test = df_test['label']

categorical_cols = ["Gender", "Race", "milk_consumption"]
numeric_cols = [col for col in X_train_raw.columns if col not in categorical_cols]

# ====== 1) Preprocess ======
preprocessor = ColumnTransformer([
    ("num", StandardScaler(), numeric_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols)
])

X_train_proc = preprocessor.fit_transform(X_train_raw)
X_test_proc = preprocessor.transform(X_test_raw)

# ====== 2) Build graph on ORIGINAL train (before SMOTE) ======
n_train = X_train_proc.shape[0]
dist_matrix = euclidean_distances(X_train_proc)
threshold = np.percentile(dist_matrix, 10)  # connect ~10% closest

G = nx.Graph()
for i in range(n_train):
    G.add_node(i)

for i in range(n_train):
    for j in range(i + 1, n_train):
        if dist_matrix[i, j] < threshold:
            G.add_edge(i, j, weight=1.0 / (1e-5 + dist_matrix[i, j]))

# ====== 3) Graph features for train ======
deg_cent = nx.degree_centrality(G)
bet_cent = nx.betweenness_centrality(G)
clust = nx.clustering(G)

graph_feats_train = pd.DataFrame({
    "id": list(G.nodes()),
    "deg_cent": [deg_cent[i] for i in G.nodes()],
    "bet_cent": [bet_cent[i] for i in G.nodes()],
    "clust": [clust[i] for i in G.nodes()]
})

# ====== 3b) Lightweight Random Walk Embedding ======
def random_walk_embedding(G, node, walk_length=10, num_walks=20, n_nodes=None):
    """Return frequency-based embedding for a node via random walks."""
    if n_nodes is None:
        n_nodes = len(G.nodes())
    freq = np.zeros(n_nodes)
    for _ in range(num_walks):
        current = node
        for _ in range(walk_length):
            neighbors = list(G.neighbors(current))
            if not neighbors:
                break
            current = random.choice(neighbors)
            freq[current] += 1
    # Normalize to probabilities
    return freq / (walk_length * num_walks)

embeddings = {node: random_walk_embedding(G, node, n_nodes=n_train) for node in G.nodes()}
embeddings_df = pd.DataFrame.from_dict(embeddings, orient="index")
embeddings_df.reset_index(inplace=True)
embeddings_df.rename(columns={"index": "id"}, inplace=True)

graph_feats_train = graph_feats_train.merge(embeddings_df, on="id")
graph_feats_train.set_index("id", inplace=True)

# ====== 4) Append graph features to train/test (before SMOTE) ======
X_train_with_graph = np.hstack([X_train_proc.toarray(), graph_feats_train.values])
X_test_with_graph = np.hstack([
    X_test_proc.toarray(),
    np.zeros((X_test_proc.shape[0], graph_feats_train.shape[1]))  # unseen test nodes
])

# ====== 5) Apply SMOTE on train with graph ======
smote = SMOTE(random_state=42)
X_train_bal, y_train_bal = smote.fit_resample(X_train_with_graph, y_train)

# ====== 6) Train classifier ======
clf = RandomForestClassifier(random_state=42)
clf.fit(X_train_bal, y_train_bal)
y_pred = clf.predict(X_test_with_graph)

print(classification_report(y_test, y_pred))


KeyboardInterrupt: 

In [27]:
# --- IMPORTS ---
import pandas as pd
import numpy as np
import networkx as nx
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics.pairwise import euclidean_distances
from sklearn.decomposition import TruncatedSVD
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from imblearn.over_sampling import SMOTE

# ====== 0) Predefined train/test ======
X_train_raw = df_train.drop(columns="label")
y_train = df_train["label"]
X_test_raw = df_test.drop(columns=["label"])
y_test = df_test["label"]
X_train_raw = X_train_raw.iloc[:100]
y_train = y_train.iloc[:100]
X_test_raw = X_test_raw.iloc[:100]
y_test = y_test.iloc[:100]


categorical_cols = ["Gender", "Race","milk_consumption"]
numeric_cols = [col for col in X_train_raw.columns if col not in categorical_cols]

# ====== 1) Preprocess ======
preprocessor = ColumnTransformer([
    ("num", StandardScaler(), numeric_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols)
])

X_train_proc = preprocessor.fit_transform(X_train_raw)
X_test_proc = preprocessor.transform(X_test_raw)

# ====== 2) Build graph on ORIGINAL train (before SMOTE) ======
n_train = X_train_proc.shape[0]
print("before dist")
dist_matrix = euclidean_distances(X_train_proc)
print("before threshold")
# connect ~10% closest pairs
threshold = np.percentile(dist_matrix, 10)

print("before graph")
G = nx.Graph()
for i in range(n_train):
    G.add_node(i)
print("before edges")
for i in range(n_train):
    for j in range(i + 1, n_train):
        if dist_matrix[i, j] < threshold:
            G.add_edge(i, j, weight=1.0 / (1e-5 + dist_matrix[i, j]))
print("before section 3")
# ====== 3) Graph features for train ======
deg_cent = nx.degree_centrality(G)
# Approximate betweenness (much faster)
bet_cent = nx.betweenness_centrality(G, k=min(100, n_train), seed=42)
clust = nx.clustering(G)

graph_feats_train = pd.DataFrame({
    "id": list(G.nodes()),
    "deg_cent": [deg_cent[i] for i in G.nodes()],
    "bet_cent": [bet_cent[i] for i in G.nodes()],
    "clust": [clust[i] for i in G.nodes()]
})

# ====== 4) Fast Graph Embedding via SVD ======
def fast_graph_embedding(G, embed_dim=32):
    """Approximate embeddings using adjacency + higher-order proximities + SVD."""
    A = nx.to_numpy_array(G)  # adjacency
    A2 = A @ A
    A3 = A2 @ A
    M = A + 0.5 * A2 + 0.25 * A3  # weighted higher-order proximity
    svd = TruncatedSVD(n_components=embed_dim, random_state=42)
    emb = svd.fit_transform(M)
    return emb
print("before fast")
embeddings = fast_graph_embedding(G, embed_dim=32)
print("after fast")
embeddings_df = pd.DataFrame(embeddings)
print("before id")
embeddings_df["id"] = list(G.nodes())
print("after id")

# Merge graph stats + embeddings
graph_feats_train = graph_feats_train.merge(embeddings_df, on="id")
graph_feats_train.set_index("id", inplace=True)

# ====== 5) Append graph features to train/test (before SMOTE) ======
# ====== 5) Append graph features to train/test (before SMOTE) ======
X_train_with_graph = np.hstack([X_train_proc, graph_feats_train.values])
X_test_with_graph = np.hstack([
    X_test_proc,
    np.zeros((X_test_proc.shape[0], graph_feats_train.shape[1]))  # test nodes not in graph
])


# ====== 6) Apply SMOTE on train with graph ======
smote = SMOTE(random_state=42)
X_train_bal, y_train_bal = smote.fit_resample(X_train_with_graph, y_train)

# ====== 7) Train classifier ======
clf = RandomForestClassifier(random_state=42)
clf.fit(X_train_bal, y_train_bal)
y_pred = clf.predict(X_test_with_graph)

print(classification_report(y_test, y_pred))


before dist
before threshold
before graph
before edges
before section 3
before fast
after fast
before id
after id


ValueError: The target 'y' needs to have more than 1 class. Got 1 class instead

In [59]:
# --- IMPORTS ---
import pandas as pd
import numpy as np
import networkx as nx
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics.pairwise import euclidean_distances
from sklearn.decomposition import TruncatedSVD
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from imblearn.over_sampling import SMOTE
from scipy.sparse import issparse  # ✅ to check sparse matrix type
from sklearn.utils import resample
# ====== 0) Predefined train/test ======
X_train_raw = df_train.drop(columns="label")
y_train = df_train["label"]
X_test_raw = df_test.drop(columns=["label"])
y_test = df_test["label"]

# Combine features + labels
train_df = pd.concat([X_train_raw, y_train], axis=1)
test_df = pd.concat([X_test_raw, y_test], axis=1)

def balanced_sample(df, label_col="label", n=100, random_state=42):
    sampled = []
    for cls in df[label_col].unique():
        cls_df = df[df[label_col] == cls]
        sampled_cls = resample(
            cls_df,
            replace=True if len(cls_df) < n else False,  # allow upsampling if not enough samples
            n_samples=n,
            random_state=random_state
        )
        sampled.append(sampled_cls)
    return pd.concat(sampled).sample(frac=1, random_state=random_state)  # shuffle

# Take 100 samples per class
train_sampled = balanced_sample(train_df, "label", n=100, random_state=42)
test_sampled = balanced_sample(test_df, "label", n=50, random_state=42)

# Split back into X, y
X_train_raw = train_sampled.drop(columns="label")
y_train = train_sampled["label"]
X_test_raw = test_sampled.drop(columns="label")
y_test = test_sampled["label"]

categorical_cols = ["Gender", "Race", "milk_consumption"]
numeric_cols = [col for col in X_train_raw.columns if col not in categorical_cols]

# ====== 1) Preprocess ======
preprocessor = ColumnTransformer([
    ("num", StandardScaler(), numeric_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=True), categorical_cols)
])

X_train_proc = preprocessor.fit_transform(X_train_raw)
X_test_proc = preprocessor.transform(X_test_raw)

# ✅ Convert to dense only if sparse
if issparse(X_train_proc):
    X_train_proc = X_train_proc.toarray()
if issparse(X_test_proc):
    X_test_proc = X_test_proc.toarray()

# ====== 2) Build graph on ORIGINAL train (before SMOTE) ======
n_train = X_train_proc.shape[0]
dist_matrix = euclidean_distances(X_train_proc)

# connect ~10% closest pairs
threshold = np.percentile(dist_matrix, 10)

G = nx.Graph()
for i in range(n_train):
    G.add_node(i)
for i in range(n_train):
    for j in range(i + 1, n_train):
        if dist_matrix[i, j] < threshold:
            G.add_edge(i, j, weight=1.0 / (1e-5 + dist_matrix[i, j]))

# ====== 3) Graph features for train ======
deg_cent = nx.degree_centrality(G)
bet_cent = nx.betweenness_centrality(G, k=min(100, n_train), seed=42)
clust = nx.clustering(G)

graph_feats_train = pd.DataFrame({
    "id": list(G.nodes()),
    "deg_cent": [deg_cent[i] for i in G.nodes()],
    "bet_cent": [bet_cent[i] for i in G.nodes()],
    "clust": [clust[i] for i in G.nodes()]
})

# ====== 4) Fast Graph Embedding via SVD ======
def fast_graph_embedding(G, embed_dim=32):
    """Approximate embeddings using adjacency + higher-order proximities + SVD."""
    A = nx.to_numpy_array(G)  # adjacency
    A2 = A @ A
    A3 = A2 @ A
    M = A + 0.5 * A2 + 0.25 * A3  # weighted higher-order proximity
    svd = TruncatedSVD(n_components=min(embed_dim, M.shape[0]-1), random_state=42)
    emb = svd.fit_transform(M)
    return emb

embeddings = fast_graph_embedding(G, embed_dim=32)
embeddings_df = pd.DataFrame(embeddings)
embeddings_df["id"] = list(G.nodes())

# Merge graph stats + embeddings
graph_feats_train = graph_feats_train.merge(embeddings_df, on="id")
graph_feats_train.set_index("id", inplace=True)

# ====== 5) Append graph features to train/test (before SMOTE) ======
X_train_with_graph = np.hstack([X_train_proc, graph_feats_train.values])
X_test_with_graph = np.hstack([
    X_test_proc,
    np.zeros((X_test_proc.shape[0], graph_feats_train.shape[1]))  # test nodes not in graph
])

# ====== 6) Apply SMOTE on train with graph ======
smote = SMOTE(random_state=42)
X_train_bal, y_train_bal = smote.fit_resample(X_train_with_graph, y_train)

# ====== 7) Train classifier ======
clf = RandomForestClassifier(random_state=42)
clf.fit(X_train_bal, y_train_bal)
y_pred = clf.predict(X_test_with_graph)

print(classification_report(y_test, y_pred))


              precision    recall  f1-score   support

         0.0       0.79      0.60      0.68        50
         1.0       0.68      0.84      0.75        50

    accuracy                           0.72       100
   macro avg       0.73      0.72      0.72       100
weighted avg       0.73      0.72      0.72       100



In [61]:
# --- IMPORTS ---
import pandas as pd
import numpy as np
import networkx as nx
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.neighbors import NearestNeighbors
from sklearn.decomposition import TruncatedSVD
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from imblearn.over_sampling import SMOTE
from scipy.sparse import issparse, csr_matrix
from sklearn.utils import resample
from sklearn.cluster import MiniBatchKMeans
import warnings
warnings.filterwarnings('ignore')

# ====== 0) Predefined train/test ======
X_train_raw = df_train.drop(columns="label")
y_train = df_train["label"]
X_test_raw = df_test.drop(columns=["label"])
y_test = df_test["label"]

# Combine features + labels
train_df = pd.concat([X_train_raw, y_train], axis=1)
test_df = pd.concat([X_test_raw, y_test], axis=1)

def balanced_sample(df, label_col="label", n=1000, random_state=42):
    """Improved balanced sampling with better memory efficiency"""
    sampled = []
    for cls in df[label_col].unique():
        cls_df = df[df[label_col] == cls]
        if len(cls_df) >= n:
            # Use pandas sample for large datasets (more memory efficient)
            sampled_cls = cls_df.sample(n=n, random_state=random_state)
        else:
            # Upsample if not enough samples
            sampled_cls = resample(
                cls_df,
                replace=True,
                n_samples=n,
                random_state=random_state
            )
        sampled.append(sampled_cls)
    return pd.concat(sampled, ignore_index=True).sample(frac=1, random_state=random_state)

# Adaptive sampling based on dataset size
n_samples_per_class = min(1000, len(train_df) // len(train_df["label"].unique()) // 2)
train_sampled = balanced_sample(train_df, "label", n=n_samples_per_class, random_state=42)
test_sampled = balanced_sample(test_df, "label", n=min(500, len(test_df) // len(test_df["label"].unique()) // 2), random_state=42)

# Split back into X, y
X_train_raw = train_sampled.drop(columns="label")
y_train = train_sampled["label"]
X_test_raw = test_sampled.drop(columns="label")
y_test = test_sampled["label"]

categorical_cols = ["Gender", "Race", "milk_consumption"]
numeric_cols = [col for col in X_train_raw.columns if col not in categorical_cols]

# ====== 1) Preprocess ======
preprocessor = ColumnTransformer([
    ("num", StandardScaler(), numeric_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=True), categorical_cols)
])

X_train_proc = preprocessor.fit_transform(X_train_raw)
X_test_proc = preprocessor.transform(X_test_raw)

# Keep sparse matrices for memory efficiency
if issparse(X_train_proc):
    X_train_dense = X_train_proc.toarray()
else:
    X_train_dense = X_train_proc
    
if issparse(X_test_proc):
    X_test_dense = X_test_proc.toarray()
else:
    X_test_dense = X_test_proc

# ====== 2) SCALABLE Graph Construction ======
def build_scalable_graph(X, k_neighbors=10, max_samples=5000, cluster_connect=True):
    """
    Build graph efficiently for large datasets using:
    1. KNN for local connections
    2. Clustering for global structure
    3. Sparse representation
    """
    n_samples = X.shape[0]
    
    # Adaptive k based on dataset size
    k = min(k_neighbors, max(3, int(np.sqrt(n_samples))))
    
    # Use subset for very large datasets to build initial structure
    if n_samples > max_samples:
        indices = np.random.choice(n_samples, max_samples, replace=False)
        X_subset = X[indices]
        print(f"Using subset of {max_samples} samples for graph construction")
    else:
        indices = np.arange(n_samples)
        X_subset = X
    
    # Build KNN graph for local connections
    nn = NearestNeighbors(n_neighbors=k+1, metric='euclidean', n_jobs=-1)
    nn.fit(X_subset)
    distances, neighbors = nn.kneighbors(X_subset)
    
    # Create sparse graph
    G = nx.Graph()
    G.add_nodes_from(range(len(indices)))
    
    # Add KNN edges
    for i in range(len(indices)):
        for j in range(1, len(neighbors[i])):  # Skip self (index 0)
            neighbor_idx = neighbors[i][j]
            if neighbor_idx != i:  # Safety check
                weight = 1.0 / (1e-6 + distances[i][j])
                G.add_edge(i, neighbor_idx, weight=weight)
    
    # Add cluster-based long-range connections for global structure
    if cluster_connect and len(indices) > 50:
        n_clusters = min(20, len(indices) // 10)
        kmeans = MiniBatchKMeans(n_clusters=n_clusters, random_state=42, batch_size=1000)
        clusters = kmeans.fit_predict(X_subset)
        
        # Connect samples within clusters (sparsely)
        for cluster_id in range(n_clusters):
            cluster_members = np.where(clusters == cluster_id)[0]
            if len(cluster_members) > 1:
                # Connect each node to 2-3 random others in same cluster
                for member in cluster_members:
                    n_connect = min(3, len(cluster_members) - 1)
                    if n_connect > 0:
                        others = np.random.choice(
                            [m for m in cluster_members if m != member], 
                            size=n_connect, replace=False
                        )
                        for other in others:
                            if not G.has_edge(member, other):
                                dist = np.linalg.norm(X_subset[member] - X_subset[other])
                                G.add_edge(member, other, weight=1.0 / (1e-6 + dist))
    
    return G, indices

# Build graph
G, graph_indices = build_scalable_graph(X_train_dense, k_neighbors=8, max_samples=3000)
n_graph_nodes = len(graph_indices)

print(f"Graph built with {G.number_of_nodes()} nodes and {G.number_of_edges()} edges")

# ====== 3) EFFICIENT Graph features ======
def compute_efficient_graph_features(G, max_nodes_for_expensive=1000):
    """Compute graph features with computational shortcuts for large graphs"""
    n_nodes = G.number_of_nodes()
    
    # Always compute these (fast)
    deg_cent = nx.degree_centrality(G)
    clust = nx.clustering(G)
    
    # Only compute expensive features for smaller graphs
    if n_nodes <= max_nodes_for_expensive:
        bet_cent = nx.betweenness_centrality(G, k=min(200, n_nodes), seed=42)
        close_cent = nx.closeness_centrality(G)
    else:
        # Use degree as proxy for betweenness (correlation often exists)
        degrees = dict(G.degree())
        max_deg = max(degrees.values()) if degrees else 1
        bet_cent = {node: degrees[node] / max_deg for node in G.nodes()}
        
        # Use clustering coefficient as proxy for closeness
        close_cent = clust.copy()
    
    # Compute local features
    triangles = nx.triangles(G)
    
    return {
        'deg_cent': deg_cent,
        'bet_cent': bet_cent,
        'close_cent': close_cent,
        'clust': clust,
        'triangles': triangles
    }

# Compute features
graph_stats = compute_efficient_graph_features(G)

# Create feature dataframe
graph_feats_train = pd.DataFrame({
    "id": list(G.nodes()),
    "deg_cent": [graph_stats['deg_cent'][i] for i in G.nodes()],
    "bet_cent": [graph_stats['bet_cent'][i] for i in G.nodes()],
    "close_cent": [graph_stats['close_cent'][i] for i in G.nodes()],
    "clust": [graph_stats['clust'][i] for i in G.nodes()],
    "triangles": [graph_stats['triangles'][i] for i in G.nodes()]
})

# ====== 4) Efficient Graph Embedding ======
def efficient_graph_embedding(G, embed_dim=16, use_laplacian=True):
    """
    More efficient embedding using:
    1. Smaller embedding dimension
    2. Laplacian eigenmaps option
    3. Sparse matrix operations
    """
    n_nodes = G.number_of_nodes()
    embed_dim = min(embed_dim, n_nodes - 1, 32)  # Cap dimension
    
    if use_laplacian and n_nodes > 100:
        # Use Laplacian eigenmaps for better spectral properties
        L = nx.normalized_laplacian_matrix(G, weight='weight')
        if issparse(L):
            svd = TruncatedSVD(n_components=embed_dim, random_state=42)
            embeddings = svd.fit_transform(L)
        else:
            # Fallback for small graphs
            eigenvals, eigenvecs = np.linalg.eigh(L.toarray())
            embeddings = eigenvecs[:, 1:embed_dim+1]  # Skip first eigenvalue
    else:
        # Original adjacency-based method for small graphs
        A = nx.to_numpy_array(G)
        if n_nodes > 10:
            A2 = A @ A
            M = A + 0.3 * A2  # Reduced weight for A2
        else:
            M = A
            
        svd = TruncatedSVD(n_components=embed_dim, random_state=42)
        embeddings = svd.fit_transform(M)
    
    return embeddings

# Generate embeddings
embeddings = efficient_graph_embedding(G, embed_dim=16)
embeddings_df = pd.DataFrame(embeddings, columns=[f'emb_{i}' for i in range(embeddings.shape[1])])
embeddings_df["id"] = list(G.nodes())

# Merge graph stats + embeddings
graph_feats_train = graph_feats_train.merge(embeddings_df, on="id")
graph_feats_train.set_index("id", inplace=True)

print(f"Graph features shape: {graph_feats_train.shape}")

# ====== 5) Extend graph features to all samples ======
def extend_graph_features_to_all(X_all, graph_features, graph_indices, method='knn'):
    """
    Extend graph features from subset to all samples using KNN imputation
    """
    n_all = X_all.shape[0]
    n_graph_features = graph_features.shape[1]
    
    # Initialize features for all samples
    extended_features = np.zeros((n_all, n_graph_features))
    
    # Direct assignment for graph nodes
    extended_features[graph_indices] = graph_features.values
    
    # Estimate for non-graph nodes using KNN
    non_graph_indices = [i for i in range(n_all) if i not in graph_indices]
    
    if len(non_graph_indices) > 0 and len(graph_indices) > 5:
        # Use KNN to find similar graph nodes
        nn = NearestNeighbors(n_neighbors=min(5, len(graph_indices)), n_jobs=-1)
        nn.fit(X_all[graph_indices])
        
        for idx in non_graph_indices:
            distances, neighbors = nn.kneighbors([X_all[idx]])
            # Weighted average of k nearest graph nodes
            weights = 1.0 / (distances[0] + 1e-6)
            weights = weights / weights.sum()
            
            neighbor_graph_indices = [graph_indices[n] for n in neighbors[0]]
            extended_features[idx] = np.average(
                graph_features.values[neighbors[0]], 
                axis=0, 
                weights=weights
            )
    
    return extended_features

# Extend to all training samples
extended_graph_features = extend_graph_features_to_all(
    X_train_dense, graph_feats_train, graph_indices
)

# For test samples, use KNN from training samples
extended_test_features = np.zeros((X_test_dense.shape[0], graph_feats_train.shape[1]))
if len(graph_indices) > 5:
    nn_test = NearestNeighbors(n_neighbors=min(5, len(graph_indices)), n_jobs=-1)
    nn_test.fit(X_train_dense[graph_indices])
    
    for i, test_sample in enumerate(X_test_dense):
        distances, neighbors = nn_test.kneighbors([test_sample])
        weights = 1.0 / (distances[0] + 1e-6)
        weights = weights / weights.sum()
        
        extended_test_features[i] = np.average(
            graph_feats_train.values[neighbors[0]], 
            axis=0, 
            weights=weights
        )

# ====== 6) Combine features ======
X_train_with_graph = np.hstack([X_train_dense, extended_graph_features])
X_test_with_graph = np.hstack([X_test_dense, extended_test_features])

print(f"Final training features shape: {X_train_with_graph.shape}")
print(f"Final test features shape: {X_test_with_graph.shape}")

# ====== 7) Apply SMOTE and train ======
smote = SVMSMOTE(random_state=42, k_neighbors=20,m_neighbors=40,sampling_strategy='minority')
X_train_bal, y_train_bal = smote.fit_resample(X_train_with_graph, y_train)

# Train classifier with optimized parameters for larger datasets
clf = RandomForestClassifier(
    n_estimators=100,
    max_depth=20,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)

print("Training classifier...")
clf.fit(X_train_bal, y_train_bal)

print("Making predictions...")
y_pred = clf.predict(X_test_with_graph)

# Results
print("\n" + "="*50)
print("CLASSIFICATION REPORT")
print("="*50)
print(classification_report(y_test, y_pred))

# Feature importance analysis
feature_names = (
    [f'num_{i}' for i in range(len(numeric_cols))] +
    [f'cat_{i}' for i in range(X_train_dense.shape[1] - len(numeric_cols))] +
    list(graph_feats_train.columns)
)

importances = clf.feature_importances_
graph_feature_importance = importances[-len(graph_feats_train.columns):].sum()
original_feature_importance = importances[:-len(graph_feats_train.columns)].sum()

print(f"\nFeature Importance Analysis:")
print(f"Original features: {original_feature_importance:.3f}")
print(f"Graph features: {graph_feature_importance:.3f}")
print(f"Graph features contribution: {graph_feature_importance/importances.sum()*100:.1f}%")

Graph built with 2000 nodes and 17232 edges
Graph features shape: (2000, 21)
Final training features shape: (2000, 43)
Final test features shape: (1000, 43)
Training classifier...
Making predictions...

CLASSIFICATION REPORT
              precision    recall  f1-score   support

         0.0       0.70      0.44      0.54       500
         1.0       0.59      0.81      0.68       500

    accuracy                           0.63      1000
   macro avg       0.65      0.63      0.61      1000
weighted avg       0.65      0.63      0.61      1000


Feature Importance Analysis:
Original features: 0.465
Graph features: 0.535
Graph features contribution: 53.5%


In [57]:
# Full updated, scalable pipeline
# --- IMPORTS ---
import pandas as pd
import numpy as np
import networkx as nx
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import TruncatedSVD
from sklearn.neighbors import NearestNeighbors
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.pipeline import make_pipeline
from imblearn.over_sampling import SMOTE
from scipy import sparse
from sklearn.utils import resample

# ================== CONFIG ==================
SAMPLE_PER_CLASS_TRAIN = None   # set int to sample per class from df_train (or None to use all)
SAMPLE_PER_CLASS_TEST = None    # set int or None
PCA_DIM = 50                    # reduce preprocessed features to this dim before SMOTE/graph
K_NEIGHBORS = 10                # k for kNN graph (sparse)
EMBED_DIM = 32                  # graph embedding dimensions (via TruncatedSVD)
BETWEENNESS_K = 100             # k for approximate betweenness centrality (set small for speed)
USE_BETWEENNESS = True          # set False to skip betweenness
RANDOM_STATE = 42
X_train_raw = df_train.drop(columns="label")
y_train = df_train["label"]
X_test_raw = df_test.drop(columns=["label"])
y_test = df_test["label"]

# Combine features + labels
train_df = pd.concat([X_train_raw, y_train], axis=1)
test_df = pd.concat([X_test_raw, y_test], axis=1)
# ================== 0) Prepare balanced sampling (optional) ==================
def balanced_sample_df(df, label_col="label", n_per_class=None, random_state=RANDOM_STATE):
    if n_per_class is None:
        return df.sample(frac=1, random_state=random_state)  # just shuffle
    parts = []
    classes = sorted(df[label_col].unique())
    for c in classes:
        d = df[df[label_col] == c]
        replace = len(d) < n_per_class
        parts.append(resample(d, replace=replace, n_samples=n_per_class, random_state=random_state))
    out = pd.concat(parts, axis=0).sample(frac=1, random_state=random_state).reset_index(drop=True)
    return out

# --- INPUT: df_train, df_test must exist ---
# If SAMPLE_PER_CLASS_* is None we keep the original (but shuffled); otherwise balanced sample per class.
train_df = pd.concat([df_train.drop(columns=[]), df_train["label"]], axis=1)
test_df  = pd.concat([df_test.drop(columns=[]), df_test["label"]], axis=1)

train_df = balanced_sample_df(train_df, "label", SAMPLE_PER_CLASS_TRAIN)
test_df  = balanced_sample_df(test_df, "label", SAMPLE_PER_CLASS_TEST)

X_train_raw = train_df.drop(columns="label").reset_index(drop=True)
y_train = train_df["label"].reset_index(drop=True)
X_test_raw = test_df.drop(columns="label").reset_index(drop=True)
y_test = test_df["label"].reset_index(drop=True)

print("Train samples:", X_train_raw.shape, "class counts:\n", y_train.value_counts())
print("Test samples:", X_test_raw.shape, "class counts:\n", y_test.value_counts())

# ================== 1) Preprocess (numeric + categorical) ==================
# Update categorical_cols to match your dataset
categorical_cols = ["Gender", "Race", "milk_consumption", "YearStart"]  # add / remove if needed
numeric_cols = [c for c in X_train_raw.columns if c not in categorical_cols]

preprocessor = ColumnTransformer([
    ("num", StandardScaler(), numeric_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse=False), categorical_cols)
], remainder="drop")

X_train_pre = preprocessor.fit_transform(X_train_raw)
X_test_pre  = preprocessor.transform(X_test_raw)

# Optionally convert to float32 for efficiency
X_train_pre = X_train_pre.astype(np.float32)
X_test_pre  = X_test_pre.astype(np.float32)

# ================== 2) Dimensionality reduction BEFORE SMOTE ==================
# Use TruncatedSVD (works on dense arrays as well)
svd_pre = TruncatedSVD(n_components=min(PCA_DIM, X_train_pre.shape[1]-1), random_state=RANDOM_STATE)
X_train_red = svd_pre.fit_transform(X_train_pre)
X_test_red  = svd_pre.transform(X_test_pre)

# ================== 3) Apply SMOTE on reduced training features ==================
# Ensure at least two classes exist
if len(np.unique(y_train)) < 2:
    raise ValueError("y_train must contain at least 2 classes for SMOTE")

smote = SMOTE(random_state=RANDOM_STATE)
X_train_bal_red, y_train_bal = smote.fit_resample(X_train_red, y_train)

print("After SMOTE - train shape:", X_train_bal_red.shape, "class counts:\n", pd.Series(y_train_bal).value_counts())

# ================== 4) Build kNN graph on train+test reduced features (so test gets graph features) ==================
# We'll build graph on concatenation: (train_balanced + test) in reduced space
X_all_for_graph = np.vstack([X_train_bal_red, X_test_red])
n_train_bal = X_train_bal_red.shape[0]
n_all = X_all_for_graph.shape[0]

nn = NearestNeighbors(n_neighbors=K_NEIGHBORS, metric="euclidean", n_jobs=-1)
nn.fit(X_all_for_graph)
# kneighbors_graph returns distances; we convert to adjacency weights (inverse dist)
knn_graph = nn.kneighbors_graph(X_all_for_graph, mode="distance")  # sparse (n_all x n_all)

# symmetrize kNN graph (make undirected)
knn_graph = 0.5 * (knn_graph + knn_graph.T)

# Convert distances -> similarity weights (avoid divide-by-zero)
# We'll transform nonzero distances to weights = 1 / (dist + eps)
eps = 1e-6
rows, cols = knn_graph.nonzero()
data = knn_graph.data
weights = 1.0 / (data + eps)
adj_sparse = sparse.csr_matrix((weights, (rows, cols)), shape=knn_graph.shape)

# ================== 5) Fast graph embedding via SVD on sparse adjacency ==================
# Use TruncatedSVD on adjacency (sparse friendly)
svd_graph = TruncatedSVD(n_components=min(EMBED_DIM, adj_sparse.shape[0]-1), random_state=RANDOM_STATE)
emb_all = svd_graph.fit_transform(adj_sparse)  # shape (n_all, EMBED_DIM)

# ================== 6) Lightweight graph structural features ==================
# Degree (from adjacency)
deg = np.array(adj_sparse.sum(axis=1)).reshape(-1)    # raw degree-like weight sum
deg_norm = deg / (adj_sparse.shape[0]-1)              # normalized

# Clustering: approximate via NetworkX on a sampled subgraph if huge; here we can compute on kNN small graph
G_sparse = nx.from_scipy_sparse_array(adj_sparse, parallel_edges=False)
clust_dict = nx.clustering(G_sparse)  # returns dict for all nodes (works for moderate sizes)
clust = np.array([clust_dict[i] for i in range(n_all)])

# Approximate betweenness (sampled) - compute on a subset of nodes if large
if USE_BETWEENNESS:
    k_for_bet = min(BETWEENNESS_K, n_all)
    # approximate betweenness using k random nodes (networkx uses these as sources)
    bet_dict = nx.betweenness_centrality(G_sparse, k=k_for_bet, seed=RANDOM_STATE)
    bet = np.array([bet_dict.get(i, 0.0) for i in range(n_all)])
else:
    bet = np.zeros(n_all)

# ================== 7) Split graph features back to train / test ==================
graph_feats_all = np.hstack([
    emb_all,
    deg_norm.reshape(-1, 1),
    clust.reshape(-1, 1),
    bet.reshape(-1, 1)
])  # shape (n_all, EMBED_DIM + 3)

graph_feats_train_bal = graph_feats_all[:n_train_bal, :]
graph_feats_test = graph_feats_all[n_train_bal:, :]

# ================== 8) Prepare final training / test arrays for classifier
# Important: classifier should train on the same preprocessed space we used for SMOTE.
# We'll use the REDUCED preprocessed features (X_train_bal_red) + graph features.
X_train_final = np.hstack([X_train_bal_red, graph_feats_train_bal])
X_test_final  = np.hstack([X_test_red, graph_feats_test])

print("Final X_train shape:", X_train_final.shape, "X_test shape:", X_test_final.shape)

# ================== 9) Train classifier (use class_weight as extra safety) ==================
clf = RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1)
clf.fit(X_train_final, y_train_bal)

# ================== 10) Evaluate on test set
y_pred = clf.predict(X_test_final)
y_prob = clf.predict_proba(X_test_final)[:, 1] if hasattr(clf, "predict_proba") else None

print("Classification report (TEST):")
print(classification_report(y_test, y_pred))
if y_prob is not None:
    try:
        print("ROC AUC (TEST):", roc_auc_score(y_test, y_prob))
    except Exception:
        pass

# ================== Notes / Tuning ==================
# - For very large n (>> 50k) reduce PCA_DIM, K_NEIGHBORS, EMBED_DIM to speed up.
# - If you prefer NO SMOTE, skip SMOTE step and use class_weight="balanced" (we keep both as options).
# - If you want graph only for a subset, build the graph on a representative subset and propagate embeddings to others (approximation).
# - If betweenness is too slow, set USE_BETWEENNESS=False.


ValueError: Grouper for 'label' not 1-dimensional

In [130]:
test_sampled.head()

NameError: name 'test_sampled' is not defined

In [42]:
train_sampled['label'].value_counts()

label
0.0    100
1.0    100
Name: count, dtype: int64

In [10]:
# --- IMPORTS ---
import pandas as pd
import numpy as np
import networkx as nx
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.neighbors import NearestNeighbors
from sklearn.decomposition import TruncatedSVD
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from imblearn.over_sampling import SMOTE
from scipy.sparse import issparse, csr_matrix
from sklearn.utils import resample
from sklearn.cluster import MiniBatchKMeans
import warnings
warnings.filterwarnings('ignore')

# ====== 0) Predefined train/test ======
X_train_raw = df_train.drop(columns="label")
y_train = df_train["label"]
X_test_raw = df_test.drop(columns=["label"])
y_test = df_test["label"]

# Combine features + labels
# train_df = pd.concat([X_train_raw, y_train], axis=1)
# test_df = pd.concat([X_test_raw, y_test], axis=1)

# def balanced_sample(df, label_col="label", n=1000, random_state=42):
#     """Improved balanced sampling with better memory efficiency"""
#     sampled = []
#     for cls in df[label_col].unique():
#         cls_df = df[df[label_col] == cls]
#         if len(cls_df) >= n:
#             # Use pandas sample for large datasets (more memory efficient)
#             sampled_cls = cls_df.sample(n=n, random_state=random_state)
#         else:
#             # Upsample if not enough samples
#             sampled_cls = resample(
#                 cls_df,
#                 replace=True,
#                 n_samples=n,
#                 random_state=random_state
#             )
#         sampled.append(sampled_cls)
#     return pd.concat(sampled, ignore_index=True).sample(frac=1, random_state=random_state)

# Adaptive sampling based on dataset size
# n_samples_per_class = min(1000, len(train_df) // len(train_df["label"].unique()) // 2)
# train_sampled = balanced_sample(train_df, "label", n=n_samples_per_class, random_state=42)
# test_sampled = balanced_sample(test_df, "label", n=min(500, len(test_df) // len(test_df["label"].unique()) // 2), random_state=42)

# Split back into X, y
# X_train_raw = train_sampled.drop(columns="label")
# y_train = train_sampled["label"]
# X_test_raw = test_sampled.drop(columns="label")
# y_test = test_sampled["label"]

categorical_cols = ["Gender", "Race", "milk_consumption"]
numeric_cols = [col for col in X_train_raw.columns if col not in categorical_cols]

# ====== 1) Preprocess ======
preprocessor = ColumnTransformer([
    ("num", StandardScaler(), numeric_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=True), categorical_cols)
])

X_train_proc = preprocessor.fit_transform(X_train_raw)
X_test_proc = preprocessor.transform(X_test_raw)

# Keep sparse matrices for memory efficiency
if issparse(X_train_proc):
    X_train_dense = X_train_proc.toarray()
else:
    X_train_dense = X_train_proc
    
if issparse(X_test_proc):
    X_test_dense = X_test_proc.toarray()
else:
    X_test_dense = X_test_proc

# ====== 2) SCALABLE Graph Construction ======
def build_scalable_graph(X, k_neighbors=10, max_samples=5000, cluster_connect=True):
    """
    Build graph efficiently for large datasets using:
    1. KNN for local connections
    2. Clustering for global structure
    3. Sparse representation
    """
    n_samples = X.shape[0]
    
    # Adaptive k based on dataset size
    k = min(k_neighbors, max(5, int(np.sqrt(n_samples))))
    
    # Use subset for very large datasets to build initial structure
    if n_samples > max_samples:
        indices = np.random.choice(n_samples, max_samples, replace=False)
        X_subset = X[indices]
        print(f"Using subset of {max_samples} samples for graph construction")
    else:
        indices = np.arange(n_samples)
        X_subset = X
    
    # Build KNN graph for local connections
    nn = NearestNeighbors(n_neighbors=k+1, metric='euclidean', n_jobs=-1)
    nn.fit(X_subset)
    distances, neighbors = nn.kneighbors(X_subset)
    
    # Create sparse graph
    G = nx.Graph()
    G.add_nodes_from(range(len(indices)))
    
    # Add KNN edges
    for i in range(len(indices)):
        for j in range(1, len(neighbors[i])):  # Skip self (index 0)
            neighbor_idx = neighbors[i][j]
            if neighbor_idx != i:  # Safety check
                weight = 1.0 / (1e-6 + distances[i][j])
                G.add_edge(i, neighbor_idx, weight=weight)
    
    # Add cluster-based long-range connections for global structure
    if cluster_connect and len(indices) > 50:
        n_clusters = min(20, len(indices) // 10)
        kmeans = MiniBatchKMeans(n_clusters=n_clusters, random_state=42, batch_size=1000)
        clusters = kmeans.fit_predict(X_subset)
        
        # Connect samples within clusters (sparsely)
        for cluster_id in range(n_clusters):
            cluster_members = np.where(clusters == cluster_id)[0]
            if len(cluster_members) > 1:
                # Connect each node to 2-3 random others in same cluster
                for member in cluster_members:
                    n_connect = min(3, len(cluster_members) - 1)
                    if n_connect > 0:
                        others = np.random.choice(
                            [m for m in cluster_members if m != member], 
                            size=n_connect, replace=False
                        )
                        for other in others:
                            if not G.has_edge(member, other):
                                dist = np.linalg.norm(X_subset[member] - X_subset[other])
                                G.add_edge(member, other, weight=1.0 / (1e-6 + dist))
    
    return G, indices

# Build graph
#k=10 ok
G, graph_indices = build_scalable_graph(X_train_dense, k_neighbors=5, max_samples=3000)
n_graph_nodes = len(graph_indices)

print(f"Graph built with {G.number_of_nodes()} nodes and {G.number_of_edges()} edges")

# ====== 3) EFFICIENT Graph features ======
def compute_efficient_graph_features(G, max_nodes_for_expensive=1000):
    """Compute graph features with computational shortcuts for large graphs"""
    n_nodes = G.number_of_nodes()
    
    # Always compute these (fast)
    deg_cent = nx.degree_centrality(G)
    clust = nx.clustering(G)
    
    # Only compute expensive features for smaller graphs
    if n_nodes <= max_nodes_for_expensive:
        bet_cent = nx.betweenness_centrality(G, k=min(200, n_nodes), seed=42)
        close_cent = nx.closeness_centrality(G)
    else:
        # Use degree as proxy for betweenness (correlation often exists)
        degrees = dict(G.degree())
        max_deg = max(degrees.values()) if degrees else 1
        bet_cent = {node: degrees[node] / max_deg for node in G.nodes()}
        
        # Use clustering coefficient as proxy for closeness
        close_cent = clust.copy()
    
    # Compute local features
    triangles = nx.triangles(G)
    
    return {
        'deg_cent': deg_cent,
        'bet_cent': bet_cent,
        'close_cent': close_cent,
        'clust': clust,
        'triangles': triangles
    }

# Compute features
graph_stats = compute_efficient_graph_features(G)

# Create feature dataframe
graph_feats_train = pd.DataFrame({
    "id": list(G.nodes()),
    "deg_cent": [graph_stats['deg_cent'][i] for i in G.nodes()],
    "bet_cent": [graph_stats['bet_cent'][i] for i in G.nodes()],
    "close_cent": [graph_stats['close_cent'][i] for i in G.nodes()],
    "clust": [graph_stats['clust'][i] for i in G.nodes()],
    "triangles": [graph_stats['triangles'][i] for i in G.nodes()]
})

# ====== 4) Efficient Graph Embedding ======
def efficient_graph_embedding(G, embed_dim=16, use_laplacian=True):
    """
    More efficient embedding using:
    1. Smaller embedding dimension
    2. Laplacian eigenmaps option
    3. Sparse matrix operations
    """
    n_nodes = G.number_of_nodes()
    embed_dim = min(embed_dim, n_nodes - 1, 32)  # Cap dimension
    
    if use_laplacian and n_nodes > 100:
        # Use Laplacian eigenmaps for better spectral properties
        L = nx.normalized_laplacian_matrix(G, weight='weight')
        if issparse(L):
            svd = TruncatedSVD(n_components=embed_dim, random_state=42)
            embeddings = svd.fit_transform(L)
        else:
            # Fallback for small graphs
            eigenvals, eigenvecs = np.linalg.eigh(L.toarray())
            embeddings = eigenvecs[:, 1:embed_dim+1]  # Skip first eigenvalue
    else:
        # Original adjacency-based method for small graphs
        A = nx.to_numpy_array(G)
        if n_nodes > 10:
            A2 = A @ A
            M = A + 0.3 * A2  # Reduced weight for A2
        else:
            M = A
            
        svd = TruncatedSVD(n_components=embed_dim, random_state=42)
        embeddings = svd.fit_transform(M)
    
    return embeddings

# Generate embeddings
embeddings = efficient_graph_embedding(G, embed_dim=20)
embeddings_df = pd.DataFrame(embeddings, columns=[f'emb_{i}' for i in range(embeddings.shape[1])])
embeddings_df["id"] = list(G.nodes())

# Merge graph stats + embeddings
graph_feats_train = graph_feats_train.merge(embeddings_df, on="id")
graph_feats_train.set_index("id", inplace=True)

print(f"Graph features shape: {graph_feats_train.shape}")

# ====== 5) Extend graph features to all samples ======
def extend_graph_features_to_all(X_all, graph_features, graph_indices, method='knn'):
    """
    Extend graph features from subset to all samples using KNN imputation
    """
    n_all = X_all.shape[0]
    n_graph_features = graph_features.shape[1]
    
    # Initialize features for all samples
    extended_features = np.zeros((n_all, n_graph_features))
    
    # Direct assignment for graph nodes
    extended_features[graph_indices] = graph_features.values
    
    # Estimate for non-graph nodes using KNN
    non_graph_indices = [i for i in range(n_all) if i not in graph_indices]
    
    if len(non_graph_indices) > 0 and len(graph_indices) > 5:
        # Use KNN to find similar graph nodes
        nn = NearestNeighbors(n_neighbors=min(5, len(graph_indices)), n_jobs=-1)
        nn.fit(X_all[graph_indices])
        
        for idx in non_graph_indices:
            distances, neighbors = nn.kneighbors([X_all[idx]])
            # Weighted average of k nearest graph nodes
            weights = 1.0 / (distances[0] + 1e-6)
            weights = weights / weights.sum()
            
            neighbor_graph_indices = [graph_indices[n] for n in neighbors[0]]
            extended_features[idx] = np.average(
                graph_features.values[neighbors[0]], 
                axis=0, 
                weights=weights
            )
    
    return extended_features

# Extend to all training samples
extended_graph_features = extend_graph_features_to_all(
    X_train_dense, graph_feats_train, graph_indices
)

# For test samples, use KNN from training samples
extended_test_features = np.zeros((X_test_dense.shape[0], graph_feats_train.shape[1]))
if len(graph_indices) > 5:
    nn_test = NearestNeighbors(n_neighbors=min(5, len(graph_indices)), n_jobs=-1)
    nn_test.fit(X_train_dense[graph_indices])
    
    for i, test_sample in enumerate(X_test_dense):
        distances, neighbors = nn_test.kneighbors([test_sample])
        weights = 1.0 / (distances[0] + 1e-6)
        weights = weights / weights.sum()
        
        extended_test_features[i] = np.average(
            graph_feats_train.values[neighbors[0]], 
            axis=0, 
            weights=weights
        )

# ====== 6) Combine features ======
X_train_with_graph = np.hstack([X_train_dense, extended_graph_features])
X_test_with_graph = np.hstack([X_test_dense, extended_test_features])

print(f"Final training features shape: {X_train_with_graph.shape}")
print(f"Final test features shape: {X_test_with_graph.shape}")

# ====== 7) Apply SMOTE and train ======
smote = SVMSMOTE(random_state=42, k_neighbors=20,m_neighbors=50,sampling_strategy='minority')
X_train_bal, y_train_bal = smote.fit_resample(X_train_with_graph, y_train)

# Train classifier with optimized parameters for larger datasets
clf = RandomForestClassifier(
    n_estimators=100,
    max_depth=20,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)
clf = LGBMClassifier(n_estimators=120, max_depth=5, num_leaves=31, learning_rate=0.05, random_state=42)
print("Training classifier...")
clf.fit(X_train_bal, y_train_bal)

print("Making predictions...")
y_pred = clf.predict(X_test_with_graph)

# Results
print("\n" + "="*50)
print("CLASSIFICATION REPORT")
print("="*50)
print(classification_report(y_test, y_pred))

# Feature importance analysis
feature_names = (
    [f'num_{i}' for i in range(len(numeric_cols))] +
    [f'cat_{i}' for i in range(X_train_dense.shape[1] - len(numeric_cols))] +
    list(graph_feats_train.columns)
)

importances = clf.feature_importances_
graph_feature_importance = importances[-len(graph_feats_train.columns):].sum()
original_feature_importance = importances[:-len(graph_feats_train.columns)].sum()

print(f"\nFeature Importance Analysis:")
print(f"Original features: {original_feature_importance:.3f}")
print(f"Graph features: {graph_feature_importance:.3f}")
print(f"Graph features contribution: {graph_feature_importance/importances.sum()*100:.1f}%")

Using subset of 3000 samples for graph construction
Graph built with 3000 nodes and 20444 edges
Graph features shape: (3000, 25)
Final training features shape: (16209, 59)
Final test features shape: (4630, 59)
Training classifier...
[LightGBM] [Info] Number of positive: 12380, number of negative: 12380
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.004632 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 14816
[LightGBM] [Info] Number of data points in the train set: 24760, number of used features: 59
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No fur

In [198]:
# ====== 7) Apply SMOTE and train ======
smote = SVMSMOTE(random_state=42, k_neighbors=100,m_neighbors=50,sampling_strategy='minority')
X_train_bal, y_train_bal = smote.fit_resample(X_train_with_graph, y_train)

# Train classifier with optimized parameters for larger datasets
clf = LGBMClassifier(n_estimators=150, max_depth=4, num_leaves=16, learning_rate=0.05, random_state=42)
print("Training classifier...")
clf.fit(X_train_bal, y_train_bal)

print("Making predictions...")
y_pred = clf.predict(X_test_with_graph)

# Results
print("\n" + "="*50)
print("CLASSIFICATION REPORT")
print("="*50)
print(classification_report(y_test, y_pred))

# Feature importance analysis
feature_names = (
    [f'num_{i}' for i in range(len(numeric_cols))] +
    [f'cat_{i}' for i in range(X_train_dense.shape[1] - len(numeric_cols))] +
    list(graph_feats_train.columns)
)

importances = clf.feature_importances_
graph_feature_importance = importances[-len(graph_feats_train.columns):].sum()
original_feature_importance = importances[:-len(graph_feats_train.columns)].sum()

print(f"\nFeature Importance Analysis:")
print(f"Original features: {original_feature_importance:.3f}")
print(f"Graph features: {graph_feature_importance:.3f}")
print(f"Graph features contribution: {graph_feature_importance/importances.sum()*100:.1f}%")

Training classifier...
Making predictions...

CLASSIFICATION REPORT
              precision    recall  f1-score   support

         0.0       0.80      0.78      0.79      3193
         1.0       0.54      0.56      0.55      1437

    accuracy                           0.71      4630
   macro avg       0.67      0.67      0.67      4630
weighted avg       0.72      0.71      0.72      4630


Feature Importance Analysis:
Original features: 1923.000
Graph features: 172.000
Graph features contribution: 8.2%


In [163]:
from imblearn.over_sampling import SMOTE, BorderlineSMOTE, KMeansSMOTE, SVMSMOTE
from sklearn.metrics import classification_report
import pandas as pd

# Danh sách các SMOTE variants
smote_variants = {
    "SMOTE": SMOTE(random_state=42, k_neighbors=5, sampling_strategy="minority"),
    "BorderlineSMOTE-1": BorderlineSMOTE(kind="borderline-1", random_state=42, k_neighbors=10, sampling_strategy="minority"),
    "BorderlineSMOTE-2": BorderlineSMOTE(kind="borderline-2", random_state=42, k_neighbors=10, sampling_strategy="minority"),
    "KMeansSMOTE": KMeansSMOTE(random_state=42, sampling_strategy="minority"),
    "SVMSMOTE": SVMSMOTE(random_state=42, k_neighbors=30, m_neighbors=50, sampling_strategy="minority")
}

results = []

for name, smote in smote_variants.items():
    print(f"\n===== Testing {name} =====")
    
    # Apply oversampling
    X_train_bal, y_train_bal = smote.fit_resample(X_train_with_graph, y_train)
    
    # Train classifier
    clf = LGBMClassifier(
        n_estimators=120, max_depth=5, num_leaves=32,
        learning_rate=0.05, random_state=42
    )
    clf.fit(X_train_bal, y_train_bal)
    
    # Predict
    y_pred = clf.predict(X_test_with_graph)
    
    # Evaluate
    report = classification_report(y_test, y_pred, output_dict=True)
    results.append({
        "SMOTE_Type": name,
        "Precision": report["weighted avg"]["precision"],
        "Recall": report["weighted avg"]["recall"],
        "F1": report["weighted avg"]["f1-score"],
        "Accuracy": report["accuracy"]
    })

# Kết quả tổng hợp
df_results = pd.DataFrame(results)
print("\n===== Summary of SMOTE Variants =====")
print(df_results)



===== Testing SMOTE =====

===== Testing BorderlineSMOTE-1 =====

===== Testing BorderlineSMOTE-2 =====

===== Testing KMeansSMOTE =====


RuntimeError: No clusters found with sufficient samples of class 1.0. Try lowering the cluster_balance_threshold or increasing the number of clusters.

In [145]:
# --- IMPORTS ---
import pandas as pd
import numpy as np
import networkx as nx
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.neighbors import NearestNeighbors
from sklearn.decomposition import TruncatedSVD
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from imblearn.over_sampling import SMOTE
from imblearn.over_sampling import SVMSMOTE
from lightgbm import LGBMClassifier
from scipy.sparse import issparse, csr_matrix
from sklearn.utils import resample
from sklearn.cluster import MiniBatchKMeans
import warnings
warnings.filterwarnings('ignore')

# ====== 0) Predefined train/test ======
X_train_raw = df_train.drop(columns="label")
y_train = df_train["label"]
X_test_raw = df_test.drop(columns=["label"])
y_test = df_test["label"]

categorical_cols = ["Gender", "Race", "milk_consumption"]
numeric_cols = [col for col in X_train_raw.columns if col not in categorical_cols]

# ====== 1) Preprocess ======
preprocessor = ColumnTransformer([
    ("num", StandardScaler(), numeric_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=True), categorical_cols)
])

X_train_proc = preprocessor.fit_transform(X_train_raw)
X_test_proc = preprocessor.transform(X_test_raw)

# Keep sparse matrices for memory efficiency
if issparse(X_train_proc):
    X_train_dense = X_train_proc.toarray()
else:
    X_train_dense = X_train_proc
    
if issparse(X_test_proc):
    X_test_dense = X_test_proc.toarray()
else:
    X_test_dense = X_test_proc

print(f"Original training data shape: {X_train_dense.shape}")
print(f"Original class distribution: {pd.Series(y_train).value_counts().to_dict()}")

# ====== 2) Apply SMOTE EARLY ======
print("Applying SMOTE to balance classes...")
smote = SMOTE(
    random_state=42, 
    k_neighbors=min(5, len(X_train_dense)-1),
    sampling_strategy='auto'  # Balance all classes
)

X_train_balanced, y_train_balanced = smote.fit_resample(X_train_dense, y_train)

print(f"Balanced training data shape: {X_train_balanced.shape}")
print(f"Balanced class distribution: {pd.Series(y_train_balanced).value_counts().to_dict()}")

# ====== 3) SCALABLE Graph Construction ======
def build_scalable_graph(X, k_neighbors=10, max_samples=5000, cluster_connect=True):
    """
    Build graph efficiently for large datasets using:
    1. KNN for local connections
    2. Clustering for global structure
    3. Sparse representation
    """
    n_samples = X.shape[0]
    
    # Adaptive k based on dataset size
    k = min(k_neighbors, max(5, int(np.sqrt(n_samples))))
    
    # Use subset for very large datasets to build initial structure
    if n_samples > max_samples:
        indices = np.random.choice(n_samples, max_samples, replace=False)
        X_subset = X[indices]
        print(f"Using subset of {max_samples} samples for graph construction")
    else:
        indices = np.arange(n_samples)
        X_subset = X
    
    # Build KNN graph for local connections
    nn = NearestNeighbors(n_neighbors=k+1, metric='euclidean', n_jobs=-1)
    nn.fit(X_subset)
    distances, neighbors = nn.kneighbors(X_subset)
    
    # Create sparse graph
    G = nx.Graph()
    G.add_nodes_from(range(len(indices)))
    
    # Add KNN edges
    for i in range(len(indices)):
        for j in range(1, len(neighbors[i])):  # Skip self (index 0)
            neighbor_idx = neighbors[i][j]
            if neighbor_idx != i:  # Safety check
                weight = 1.0 / (1e-6 + distances[i][j])
                G.add_edge(i, neighbor_idx, weight=weight)
    
    # Add cluster-based long-range connections for global structure
    if cluster_connect and len(indices) > 50:
        n_clusters = min(20, len(indices) // 10)
        kmeans = MiniBatchKMeans(n_clusters=n_clusters, random_state=42, batch_size=1000)
        clusters = kmeans.fit_predict(X_subset)
        
        # Connect samples within clusters (sparsely)
        for cluster_id in range(n_clusters):
            cluster_members = np.where(clusters == cluster_id)[0]
            if len(cluster_members) > 1:
                # Connect each node to 2-3 random others in same cluster
                for member in cluster_members:
                    n_connect = min(3, len(cluster_members) - 1)
                    if n_connect > 0:
                        others = np.random.choice(
                            [m for m in cluster_members if m != member], 
                            size=n_connect, replace=False
                        )
                        for other in others:
                            if not G.has_edge(member, other):
                                dist = np.linalg.norm(X_subset[member] - X_subset[other])
                                G.add_edge(member, other, weight=1.0 / (1e-6 + dist))
    
    return G, indices

# Build graph on BALANCED data
print("Building graph on balanced training data...")
G, graph_indices = build_scalable_graph(X_train_balanced, k_neighbors=30, max_samples=4000)
n_graph_nodes = len(graph_indices)

print(f"Graph built with {G.number_of_nodes()} nodes and {G.number_of_edges()} edges")

# ====== 4) EFFICIENT Graph features ======
def compute_efficient_graph_features(G, max_nodes_for_expensive=1000):
    """Compute graph features with computational shortcuts for large graphs"""
    n_nodes = G.number_of_nodes()
    
    # Always compute these (fast)
    deg_cent = nx.degree_centrality(G)
    clust = nx.clustering(G)
    
    # Only compute expensive features for smaller graphs
    if n_nodes <= max_nodes_for_expensive:
        bet_cent = nx.betweenness_centrality(G, k=min(200, n_nodes), seed=42)
        close_cent = nx.closeness_centrality(G)
    else:
        # Use degree as proxy for betweenness (correlation often exists)
        degrees = dict(G.degree())
        max_deg = max(degrees.values()) if degrees else 1
        bet_cent = {node: degrees[node] / max_deg for node in G.nodes()}
        
        # Use clustering coefficient as proxy for closeness
        close_cent = clust.copy()
    
    # Compute local features
    triangles = nx.triangles(G)
    
    return {
        'deg_cent': deg_cent,
        'bet_cent': bet_cent,
        'close_cent': close_cent,
        'clust': clust,
        'triangles': triangles
    }

# Compute features
print("Computing graph features...")
graph_stats = compute_efficient_graph_features(G)

# Create feature dataframe
graph_feats_train = pd.DataFrame({
    "id": list(G.nodes()),
    "deg_cent": [graph_stats['deg_cent'][i] for i in G.nodes()],
    "bet_cent": [graph_stats['bet_cent'][i] for i in G.nodes()],
    "close_cent": [graph_stats['close_cent'][i] for i in G.nodes()],
    "clust": [graph_stats['clust'][i] for i in G.nodes()],
    "triangles": [graph_stats['triangles'][i] for i in G.nodes()]
})

# ====== 5) Efficient Graph Embedding ======
def efficient_graph_embedding(G, embed_dim=16, use_laplacian=True):
    """
    More efficient embedding using:
    1. Smaller embedding dimension
    2. Laplacian eigenmaps option
    3. Sparse matrix operations
    """
    n_nodes = G.number_of_nodes()
    embed_dim = min(embed_dim, n_nodes - 1, 32)  # Cap dimension
    
    if use_laplacian and n_nodes > 100:
        # Use Laplacian eigenmaps for better spectral properties
        L = nx.normalized_laplacian_matrix(G, weight='weight')
        if issparse(L):
            svd = TruncatedSVD(n_components=embed_dim, random_state=42)
            embeddings = svd.fit_transform(L)
        else:
            # Fallback for small graphs
            eigenvals, eigenvecs = np.linalg.eigh(L.toarray())
            embeddings = eigenvecs[:, 1:embed_dim+1]  # Skip first eigenvalue
    else:
        # Original adjacency-based method for small graphs
        A = nx.to_numpy_array(G)
        if n_nodes > 10:
            A2 = A @ A
            M = A + 0.3 * A2  # Reduced weight for A2
        else:
            M = A
            
        svd = TruncatedSVD(n_components=embed_dim, random_state=42)
        embeddings = svd.fit_transform(M)
    
    return embeddings

# Generate embeddings
print("Generating graph embeddings...")
embeddings = efficient_graph_embedding(G, embed_dim=16)
embeddings_df = pd.DataFrame(embeddings, columns=[f'emb_{i}' for i in range(embeddings.shape[1])])
embeddings_df["id"] = list(G.nodes())

# Merge graph stats + embeddings
graph_feats_train = graph_feats_train.merge(embeddings_df, on="id")
graph_feats_train.set_index("id", inplace=True)

print(f"Graph features shape: {graph_feats_train.shape}")

# ====== 6) Extend graph features to all samples ======
def extend_graph_features_to_all(X_all, graph_features, graph_indices, method='knn'):
    """
    Extend graph features from subset to all samples using KNN imputation
    """
    n_all = X_all.shape[0]
    n_graph_features = graph_features.shape[1]
    
    # Initialize features for all samples
    extended_features = np.zeros((n_all, n_graph_features))
    
    # Direct assignment for graph nodes
    extended_features[graph_indices] = graph_features.values
    
    # Estimate for non-graph nodes using KNN
    non_graph_indices = [i for i in range(n_all) if i not in graph_indices]
    
    if len(non_graph_indices) > 0 and len(graph_indices) > 5:
        # Use KNN to find similar graph nodes
        nn = NearestNeighbors(n_neighbors=min(5, len(graph_indices)), n_jobs=-1)
        nn.fit(X_all[graph_indices])
        
        for idx in non_graph_indices:
            distances, neighbors = nn.kneighbors([X_all[idx]])
            # Weighted average of k nearest graph nodes
            weights = 1.0 / (distances[0] + 1e-6)
            weights = weights / weights.sum()
            
            neighbor_graph_indices = [graph_indices[n] for n in neighbors[0]]
            extended_features[idx] = np.average(
                graph_features.values[neighbors[0]], 
                axis=0, 
                weights=weights
            )
    
    return extended_features

# Extend to all BALANCED training samples
print("Extending graph features to all training samples...")
extended_graph_features = extend_graph_features_to_all(
    X_train_balanced, graph_feats_train, graph_indices
)

# For test samples, use KNN from BALANCED training samples
print("Computing graph features for test samples...")
extended_test_features = np.zeros((X_test_dense.shape[0], graph_feats_train.shape[1]))
if len(graph_indices) > 5:
    nn_test = NearestNeighbors(n_neighbors=min(5, len(graph_indices)), n_jobs=-1)
    nn_test.fit(X_train_balanced[graph_indices])
    
    for i, test_sample in enumerate(X_test_dense):
        distances, neighbors = nn_test.kneighbors([test_sample])
        weights = 1.0 / (distances[0] + 1e-6)
        weights = weights / weights.sum()
        
        extended_test_features[i] = np.average(
            graph_feats_train.values[neighbors[0]], 
            axis=0, 
            weights=weights
        )

# ====== 7) Combine features ======
X_train_with_graph = np.hstack([X_train_balanced, extended_graph_features])
X_test_with_graph = np.hstack([X_test_dense, extended_test_features])

print(f"Final training features shape: {X_train_with_graph.shape}")
print(f"Final test features shape: {X_test_with_graph.shape}")

# ====== 8) Train classifier (NO additional SMOTE needed) ======
# Train classifier with optimized parameters for larger datasets
clf = LGBMClassifier(
    n_estimators=120, 
    max_depth=5, 
    num_leaves=31, 
    learning_rate=0.05, 
    random_state=42,
    verbose=-1
)

print("Training classifier on balanced data with graph features...")
clf.fit(X_train_with_graph, y_train_balanced)

print("Making predictions...")
y_pred = clf.predict(X_test_with_graph)

# Results
print("\n" + "="*50)
print("CLASSIFICATION REPORT")
print("="*50)
print(classification_report(y_test, y_pred))

# Feature importance analysis
feature_names = (
    [f'num_{i}' for i in range(len(numeric_cols))] +
    [f'cat_{i}' for i in range(X_train_balanced.shape[1] - len(numeric_cols))] +
    list(graph_feats_train.columns)
)

importances = clf.feature_importances_
graph_feature_importance = importances[-len(graph_feats_train.columns):].sum()
original_feature_importance = importances[:-len(graph_feats_train.columns)].sum()

print(f"\nFeature Importance Analysis:")
print(f"Original features: {original_feature_importance:.3f}")
print(f"Graph features: {graph_feature_importance:.3f}")
print(f"Graph features contribution: {graph_feature_importance/importances.sum()*100:.1f}%")

# Additional analysis
print(f"\nDataset Statistics:")
print(f"Original training size: {X_train_dense.shape[0]}")
print(f"Balanced training size: {X_train_balanced.shape[0]}")
print(f"Test size: {X_test_dense.shape[0]}")
print(f"Graph nodes used: {len(graph_indices)}")
print(f"Total features (original + graph): {X_train_with_graph.shape[1]}")

Original training data shape: (16209, 22)
Original class distribution: {0.0: 12380, 1.0: 3829}
Applying SMOTE to balance classes...
Balanced training data shape: (24760, 22)
Balanced class distribution: {0.0: 12380, 1.0: 12380}
Building graph on balanced training data...
Using subset of 4000 samples for graph construction
Graph built with 4000 nodes and 96307 edges
Computing graph features...
Generating graph embeddings...
Graph features shape: (4000, 21)
Extending graph features to all training samples...
Computing graph features for test samples...
Final training features shape: (24760, 43)
Final test features shape: (4630, 43)
Training classifier on balanced data with graph features...
Making predictions...

CLASSIFICATION REPORT
              precision    recall  f1-score   support

         0.0       0.83      0.64      0.72      3193
         1.0       0.47      0.71      0.56      1437

    accuracy                           0.66      4630
   macro avg       0.65      0.67      

In [ ]:
X_TRAIN_BAL = X_train_bal.copy()
Y_TRAIN_BAL = y_train_bal.copy()
classifiers = {
    'LightGBM': LGBMClassifier(n_estimators=120, max_depth=6, num_leaves=31, learning_rate=0.05, random_state=42),
    'XGBoost': XGBClassifier(n_estimators=80, learning_rate=0.066, random_state=42, verbosity=0),
    'GradiantBoosting': GradientBoostingClassifier(n_estimators=80, learning_rate=0.05, random_state=42),
    'RandomForest': RandomForestClassifier(n_estimators=100, random_state=42),
    'GradientBoosting': GradientBoostingClassifier(n_estimators=100, learning_rate=0.05, random_state=42),
    'Naive Bayes': GaussianNB(var_smoothing=1e-10),
    'SVM': SVC(kernel='rbf', C=10, gamma=0.01, class_weight='balanced', probability=True, random_state=42),
    'Stacking':stacking_clf
}
MLPClassifier(
    hidden_layer_sizes=(200, 100, 50),  # 3 hidden layers
    activation='relu',
    solver='adam',
    alpha=0.001,  # L2 regularization
    batch_size='auto',
    learning_rate='adaptive',
    learning_rate_init=0.001,
    max_iter=500,
    shuffle=True,
    random_state=42,
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=10,
    tol=1e-4
)

In [130]:

clf = GradientBoostingClassifier(n_estimators=100, learning_rate=0.05, random_state=42)
print("Training classifier...")
clf.fit(X_train_bal, y_train_bal)

print("Making predictions...")
y_pred = clf.predict(X_test_with_graph)

# Results
print("\n" + "="*50)
print("CLASSIFICATION REPORT")
print("="*50)
print(classification_report(y_test, y_pred))

# Feature importance analysis
feature_names = (
    [f'num_{i}' for i in range(len(numeric_cols))] +
    [f'cat_{i}' for i in range(X_train_dense.shape[1] - len(numeric_cols))] +
    list(graph_feats_train.columns)
)

importances = clf.feature_importances_
graph_feature_importance = importances[-len(graph_feats_train.columns):].sum()
original_feature_importance = importances[:-len(graph_feats_train.columns)].sum()

print(f"\nFeature Importance Analysis:")
print(f"Original features: {original_feature_importance:.3f}")
print(f"Graph features: {graph_feature_importance:.3f}")
print(f"Graph features contribution: {graph_feature_importance/importances.sum()*100:.1f}%")

Training classifier...
Making predictions...

CLASSIFICATION REPORT
              precision    recall  f1-score   support

         0.0       0.78      0.80      0.79      3193
         1.0       0.53      0.51      0.52      1437

    accuracy                           0.71      4630
   macro avg       0.66      0.65      0.66      4630
weighted avg       0.71      0.71      0.71      4630


Feature Importance Analysis:
Original features: 0.961
Graph features: 0.039
Graph features contribution: 3.9%


In [115]:
# --- IMPORTS ---
import pandas as pd
import numpy as np
import networkx as nx
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.neighbors import NearestNeighbors
from sklearn.decomposition import TruncatedSVD
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from imblearn.over_sampling import SMOTE
from scipy.sparse import issparse, csr_matrix
from sklearn.utils import resample
from sklearn.cluster import MiniBatchKMeans
import warnings
warnings.filterwarnings('ignore')

# ====== 0) Predefined train/test ======
X_train_raw = df_train.drop(columns="label")
y_train = df_train["label"]
X_test_raw = df_test.drop(columns=["label"])
y_test = df_test["label"]

# Combine features + labels
train_df = pd.concat([X_train_raw, y_train], axis=1)
test_df = pd.concat([X_test_raw, y_test], axis=1)

def balanced_sample(df, label_col="label", n=1000, random_state=42):
    """Improved balanced sampling with better memory efficiency"""
    sampled = []
    for cls in df[label_col].unique():
        cls_df = df[df[label_col] == cls]
        if len(cls_df) >= n:
            # Use pandas sample for large datasets (more memory efficient)
            sampled_cls = cls_df.sample(n=n, random_state=random_state)
        else:
            # Upsample if not enough samples
            sampled_cls = resample(
                cls_df,
                replace=True,
                n_samples=n,
                random_state=random_state
            )
        sampled.append(sampled_cls)
    return pd.concat(sampled, ignore_index=True).sample(frac=1, random_state=random_state)

# Adaptive sampling based on dataset size
n_samples_per_class = min(1000, len(train_df) // len(train_df["label"].unique()) // 2)
train_sampled = balanced_sample(train_df, "label", n=n_samples_per_class, random_state=42)
test_sampled = balanced_sample(test_df, "label", n=min(500, len(test_df) // len(test_df["label"].unique()) // 2), random_state=42)

# Split back into X, y
X_train_raw = train_sampled.drop(columns="label")
y_train = train_sampled["label"]
X_test_raw = test_sampled.drop(columns="label")
y_test = test_sampled["label"]

categorical_cols = ["Gender", "Race", "milk_consumption"]
numeric_cols = [col for col in X_train_raw.columns if col not in categorical_cols]

print(f"Class distribution in training: {y_train.value_counts().to_dict()}")
print(f"Class distribution in test: {y_test.value_counts().to_dict()}")

# ====== 1) Preprocess ======
preprocessor = ColumnTransformer([
    ("num", StandardScaler(), numeric_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=True), categorical_cols)
])

X_train_proc = preprocessor.fit_transform(X_train_raw)
X_test_proc = preprocessor.transform(X_test_raw)

# Keep sparse matrices for memory efficiency
if issparse(X_train_proc):
    X_train_dense = X_train_proc.toarray()
else:
    X_train_dense = X_train_proc
    
if issparse(X_test_proc):
    X_test_dense = X_test_proc.toarray()
else:
    X_test_dense = X_test_proc

# ====== 2) CLASS-AWARE SCALABLE Graph Construction ======
def build_class_aware_graph(X, y, k_neighbors=10, max_samples=5000, 
                           intra_class_weight=1.0, inter_class_weight=0.3,
                           minority_boost=2.0):
    """
    Build graph with class-aware connections to prevent majority class dominance:
    1. Stronger connections within same class (especially minority)
    2. Weaker but important cross-class connections
    3. Balanced sampling from each class for graph construction
    """
    n_samples = X.shape[0]
    classes = np.unique(y)
    
    # Ensure balanced representation in graph subset
    if n_samples > max_samples:
        # Sample proportionally from each class, with minimum representation
        indices_by_class = {}
        selected_indices = []
        
        # Calculate samples per class (ensure minority class has enough representation)
        class_counts = {cls: np.sum(y == cls) for cls in classes}
        minority_class = min(class_counts.keys(), key=lambda k: class_counts[k])
        
        # Allocate samples per class
        samples_per_class = max_samples // len(classes)
        min_minority_samples = min(300, max_samples // 4)  # Guarantee minority representation
        
        for cls in classes:
            class_indices = np.where(y == cls)[0]
            if cls == minority_class:
                n_class_samples = max(min_minority_samples, samples_per_class)
            else:
                n_class_samples = samples_per_class
                
            if len(class_indices) >= n_class_samples:
                selected = np.random.choice(class_indices, n_class_samples, replace=False)
            else:
                selected = class_indices  # Use all available
                
            indices_by_class[cls] = selected
            selected_indices.extend(selected)
        
        indices = np.array(selected_indices)
        X_subset = X[indices]
        y_subset = y[indices]
        print(f"Using balanced subset: {len(indices)} samples")
        print(f"Subset class distribution: {pd.Series(y_subset).value_counts().to_dict()}")
    else:
        indices = np.arange(n_samples)
        X_subset = X
        y_subset = y
        indices_by_class = {cls: np.where(y_subset == cls)[0] for cls in classes}
    
    # Adaptive k based on dataset size and class balance
    k = min(k_neighbors, max(3, int(np.sqrt(len(indices)))))
    
    # Create graph
    G = nx.Graph()
    G.add_nodes_from(range(len(indices)))
    
    # Add node attributes (class labels)
    node_classes = {i: y_subset[i] for i in range(len(indices))}
    nx.set_node_attributes(G, node_classes, 'class')
    
    # Build separate KNN for each class (stronger intra-class connections)
    for cls in classes:
        cls_mask = (y_subset == cls)
        if np.sum(cls_mask) > 1:  # Need at least 2 samples
            cls_indices = np.where(cls_mask)[0]
            X_cls = X_subset[cls_mask]
            
            # Intra-class connections (stronger for minority class)
            k_cls = min(k, len(cls_indices) - 1)
            if k_cls > 0:
                nn_cls = NearestNeighbors(n_neighbors=k_cls + 1, metric='euclidean', n_jobs=-1)
                nn_cls.fit(X_cls)
                distances, neighbors = nn_cls.kneighbors(X_cls)
                
                # Weight boost for minority class
                weight_multiplier = minority_boost if cls == minority_class else 1.0
                
                for i, global_i in enumerate(cls_indices):
                    for j in range(1, len(neighbors[i])):  # Skip self
                        global_j = cls_indices[neighbors[i][j]]
                        if global_i != global_j:
                            weight = (intra_class_weight * weight_multiplier) / (1e-6 + distances[i][j])
                            G.add_edge(global_i, global_j, weight=weight, type='intra')
    
    # Add strategic inter-class connections (prevent class isolation)
    for cls1 in classes:
        for cls2 in classes:
            if cls1 < cls2:  # Avoid duplicates
                cls1_indices = np.where(y_subset == cls1)[0]
                cls2_indices = np.where(y_subset == cls2)[0]
                
                if len(cls1_indices) > 0 and len(cls2_indices) > 0:
                    # Find nearest cross-class neighbors (fewer connections)
                    nn_cross = NearestNeighbors(n_neighbors=min(3, len(cls2_indices)), 
                                              metric='euclidean', n_jobs=-1)
                    nn_cross.fit(X_subset[cls2_indices])
                    
                    distances, neighbors = nn_cross.kneighbors(X_subset[cls1_indices])
                    
                    for i, global_i in enumerate(cls1_indices):
                        for j in range(min(2, len(neighbors[i]))):  # Only closest 1-2 connections
                            global_j = cls2_indices[neighbors[i][j]]
                            weight = inter_class_weight / (1e-6 + distances[i][j])
                            G.add_edge(global_i, global_j, weight=weight, type='inter')
    
    # Add cluster-based connections within each class for global structure
    for cls in classes:
        cls_mask = (y_subset == cls)
        cls_indices = np.where(cls_mask)[0]
        
        if len(cls_indices) > 10:  # Only if enough samples
            X_cls = X_subset[cls_mask]
            n_clusters = min(5, len(cls_indices) // 5)
            
            if n_clusters > 1:
                kmeans = MiniBatchKMeans(n_clusters=n_clusters, random_state=42, batch_size=100)
                clusters = kmeans.fit_predict(X_cls)
                
                # Connect samples within clusters
                for cluster_id in range(n_clusters):
                    cluster_members = cls_indices[clusters == cluster_id]
                    if len(cluster_members) > 1:
                        # Connect each node to 1-2 random others in same cluster
                        for member in cluster_members:
                            n_connect = min(2, len(cluster_members) - 1)
                            if n_connect > 0:
                                others = np.random.choice(
                                    [m for m in cluster_members if m != member], 
                                    size=n_connect, replace=False
                                )
                                for other in others:
                                    if not G.has_edge(member, other):
                                        dist = np.linalg.norm(X_subset[member] - X_subset[other])
                                        weight_mult = minority_boost if cls == minority_class else 1.0
                                        weight = (weight_mult * 0.5) / (1e-6 + dist)
                                        G.add_edge(member, other, weight=weight, type='cluster')
    
    return G, indices, y_subset

# Build class-aware graph
G, graph_indices, y_graph = build_class_aware_graph(
    X_train_dense, y_train, 
    k_neighbors=8, 
    max_samples=3000,
    intra_class_weight=1.0,
    inter_class_weight=0.4,  # Increased to prevent isolation
    minority_boost=1.5
)

print(f"Graph built with {G.number_of_nodes()} nodes and {G.number_of_edges()} edges")

# Analyze graph connectivity by class
intra_edges = len([e for e in G.edges(data=True) if e[2].get('type') == 'intra'])
inter_edges = len([e for e in G.edges(data=True) if e[2].get('type') == 'inter'])
cluster_edges = len([e for e in G.edges(data=True) if e[2].get('type') == 'cluster'])

print(f"Edge types - Intra-class: {intra_edges}, Inter-class: {inter_edges}, Cluster: {cluster_edges}")

# ====== 3) CLASS-AWARE Graph Features ======
def compute_class_aware_graph_features(G, node_classes, max_nodes_for_expensive=1000):
    """Compute graph features that consider class structure"""
    n_nodes = G.number_of_nodes()
    
    # Standard features
    deg_cent = nx.degree_centrality(G)
    clust = nx.clustering(G)
    
    # Class-specific features
    intra_class_degree = {}
    inter_class_degree = {}
    class_homophily = {}
    
    for node in G.nodes():
        node_class = node_classes.get(node, 0)
        neighbors = list(G.neighbors(node))
        
        intra_neighbors = sum(1 for n in neighbors if node_classes.get(n, 0) == node_class)
        inter_neighbors = len(neighbors) - intra_neighbors
        
        intra_class_degree[node] = intra_neighbors / max(1, len(neighbors))
        inter_class_degree[node] = inter_neighbors / max(1, len(neighbors))
        class_homophily[node] = intra_neighbors / max(1, intra_neighbors + inter_neighbors)
    
    # Expensive features only for smaller graphs
    if n_nodes <= max_nodes_for_expensive:
        bet_cent = nx.betweenness_centrality(G, k=min(200, n_nodes), seed=42)
        close_cent = nx.closeness_centrality(G)
    else:
        # Use degree as proxy, but weight by class diversity
        degrees = dict(G.degree())
        max_deg = max(degrees.values()) if degrees else 1
        bet_cent = {node: (degrees[node] / max_deg) * (1 + inter_class_degree[node]) 
                   for node in G.nodes()}
        close_cent = {node: clust[node] * (1 + intra_class_degree[node]) 
                     for node in G.nodes()}
    
    triangles = nx.triangles(G)
    
    return {
        'deg_cent': deg_cent,
        'bet_cent': bet_cent,
        'close_cent': close_cent,
        'clust': clust,
        'triangles': triangles,
        'intra_class_degree': intra_class_degree,
        'inter_class_degree': inter_class_degree,
        'class_homophily': class_homophily
    }

# Compute class-aware features
node_classes = {i: y_graph[i] for i in range(len(y_graph))}
graph_stats = compute_class_aware_graph_features(G, node_classes)

# Create enhanced feature dataframe
graph_feats_train = pd.DataFrame({
    "id": list(G.nodes()),
    "deg_cent": [graph_stats['deg_cent'][i] for i in G.nodes()],
    "bet_cent": [graph_stats['bet_cent'][i] for i in G.nodes()],
    "close_cent": [graph_stats['close_cent'][i] for i in G.nodes()],
    "clust": [graph_stats['clust'][i] for i in G.nodes()],
    "triangles": [graph_stats['triangles'][i] for i in G.nodes()],
    "intra_class_deg": [graph_stats['intra_class_degree'][i] for i in G.nodes()],
    "inter_class_deg": [graph_stats['inter_class_degree'][i] for i in G.nodes()],
    "class_homophily": [graph_stats['class_homophily'][i] for i in G.nodes()]
})

# ====== 4) Class-aware Graph Embedding ======
def class_aware_graph_embedding(G, node_classes, embed_dim=16):
    """
    Generate embeddings that preserve both proximity and class structure
    """
    n_nodes = G.number_of_nodes()
    embed_dim = min(embed_dim, n_nodes - 1, 24)
    
    if n_nodes > 100:
        # Create class-weighted adjacency matrix
        A = nx.to_numpy_array(G, weight='weight')
        
        # Add class-based regularization
        classes = list(set(node_classes.values()))
        for cls in classes:
            cls_nodes = [i for i, node in enumerate(G.nodes()) if node_classes[node] == cls]
            if len(cls_nodes) > 1:
                # Boost within-class connections slightly in embedding
                for i in cls_nodes:
                    for j in cls_nodes:
                        if i != j and A[i, j] > 0:
                            A[i, j] *= 1.1  # Small boost
        
        # Use normalized Laplacian for better embedding
        D = np.diag(np.sum(A, axis=1))
        D_inv_sqrt = np.diag(1.0 / np.sqrt(np.diag(D) + 1e-6))
        L_norm = np.eye(n_nodes) - D_inv_sqrt @ A @ D_inv_sqrt
        
        # SVD embedding
        svd = TruncatedSVD(n_components=embed_dim, random_state=42)
        embeddings = svd.fit_transform(L_norm)
    else:
        # Simple adjacency embedding for small graphs
        A = nx.to_numpy_array(G, weight='weight')
        svd = TruncatedSVD(n_components=embed_dim, random_state=42)
        embeddings = svd.fit_transform(A)
    
    return embeddings

# Generate class-aware embeddings
embeddings = class_aware_graph_embedding(G, node_classes, embed_dim=16)
embeddings_df = pd.DataFrame(embeddings, columns=[f'emb_{i}' for i in range(embeddings.shape[1])])
embeddings_df["id"] = list(G.nodes())

# Merge all features
graph_feats_train = graph_feats_train.merge(embeddings_df, on="id")
graph_feats_train.set_index("id", inplace=True)

print(f"Graph features shape: {graph_feats_train.shape}")

# ====== 5) Class-aware feature extension ======
def extend_graph_features_class_aware(X_all, y_all, graph_features, graph_indices, y_graph):
    """
    Extend graph features considering class information for better imputation
    """
    n_all = X_all.shape[0]
    n_graph_features = graph_features.shape[1]
    
    # Initialize features
    extended_features = np.zeros((n_all, n_graph_features))
    
    # Direct assignment for graph nodes
    extended_features[graph_indices] = graph_features.values
    
    # Class-aware imputation for non-graph nodes
    non_graph_indices = [i for i in range(n_all) if i not in graph_indices]
    
    if len(non_graph_indices) > 0 and len(graph_indices) > 5:
        # Separate KNN models for each class
        classes = np.unique(y_all)
        
        for target_class in classes:
            # Get non-graph samples of this class
            target_indices = [i for i in non_graph_indices if y_all[i] == target_class]
            
            if len(target_indices) > 0:
                # Find graph nodes of the same class
                same_class_graph_mask = [i for i, idx in enumerate(graph_indices) 
                                       if y_graph[i] == target_class]
                
                if len(same_class_graph_mask) >= 3:
                    # Use same-class graph nodes for imputation
                    same_class_graph_indices = [graph_indices[i] for i in same_class_graph_mask]
                    same_class_features = graph_features.values[same_class_graph_mask]
                    
                    nn = NearestNeighbors(n_neighbors=min(5, len(same_class_graph_mask)), n_jobs=-1)
                    nn.fit(X_all[same_class_graph_indices])
                    
                    for idx in target_indices:
                        distances, neighbors = nn.kneighbors([X_all[idx]])
                        weights = 1.0 / (distances[0] + 1e-6)
                        weights = weights / weights.sum()
                        
                        extended_features[idx] = np.average(
                            same_class_features[neighbors[0]], 
                            axis=0, 
                            weights=weights
                        )
                else:
                    # Fallback to all graph nodes if not enough same-class nodes
                    nn = NearestNeighbors(n_neighbors=min(5, len(graph_indices)), n_jobs=-1)
                    nn.fit(X_all[graph_indices])
                    
                    for idx in target_indices:
                        distances, neighbors = nn.kneighbors([X_all[idx]])
                        weights = 1.0 / (distances[0] + 1e-6)
                        weights = weights / weights.sum()
                        
                        extended_features[idx] = np.average(
                            graph_features.values[neighbors[0]], 
                            axis=0, 
                            weights=weights
                        )
    
    return extended_features

# Extend features class-aware
extended_graph_features = extend_graph_features_class_aware(
    X_train_dense, y_train, graph_feats_train, graph_indices, y_graph
)

# For test samples - class-aware extension
extended_test_features = np.zeros((X_test_dense.shape[0], graph_feats_train.shape[1]))
if len(graph_indices) > 5:
    classes = np.unique(y_test)
    
    for target_class in classes:
        test_class_indices = np.where(y_test == target_class)[0]
        
        if len(test_class_indices) > 0:
            # Find graph nodes of same class
            same_class_graph_mask = [i for i, idx in enumerate(graph_indices) 
                                   if y_graph[i] == target_class]
            
            if len(same_class_graph_mask) >= 3:
                same_class_graph_indices = [graph_indices[i] for i in same_class_graph_mask]
                same_class_features = graph_feats_train.values[same_class_graph_mask]
                
                nn_test = NearestNeighbors(n_neighbors=min(5, len(same_class_graph_mask)), n_jobs=-1)
                nn_test.fit(X_train_dense[same_class_graph_indices])
                
                for i in test_class_indices:
                    distances, neighbors = nn_test.kneighbors([X_test_dense[i]])
                    weights = 1.0 / (distances[0] + 1e-6)
                    weights = weights / weights.sum()
                    
                    extended_test_features[i] = np.average(
                        same_class_features[neighbors[0]], 
                        axis=0, 
                        weights=weights
                    )
            else:
                # Fallback to all graph nodes
                nn_test = NearestNeighbors(n_neighbors=min(5, len(graph_indices)), n_jobs=-1)
                nn_test.fit(X_train_dense[graph_indices])
                
                for i in test_class_indices:
                    distances, neighbors = nn_test.kneighbors([X_test_dense[i]])
                    weights = 1.0 / (distances[0] + 1e-6)
                    weights = weights / weights.sum()
                    
                    extended_test_features[i] = np.average(
                        graph_feats_train.values[neighbors[0]], 
                        axis=0, 
                        weights=weights
                    )

# ====== 6) Combine features ======
X_train_with_graph = np.hstack([X_train_dense, extended_graph_features])
X_test_with_graph = np.hstack([X_test_dense, extended_test_features])

print(f"Final training features shape: {X_train_with_graph.shape}")
print(f"Final test features shape: {X_test_with_graph.shape}")

# ====== 7) Class-balanced training ======
# Use stratified SMOTE for better balance
smote = SMOTE(random_state=42, k_neighbors=min(5, len(X_train_with_graph)-1))
X_train_bal, y_train_bal = smote.fit_resample(X_train_with_graph, y_train)

print(f"After SMOTE - Class distribution: {pd.Series(y_train_bal).value_counts().to_dict()}")

# Train classifier with class-balanced parameters
class_weights = {0: 1.0, 1: 1.0}  # Start with balanced weights
clf = RandomForestClassifier(
    n_estimators=150,
    max_depth=25,
    min_samples_split=4,
    min_samples_leaf=2,
    class_weight=class_weights,  # Balanced class weights
    random_state=42,
    n_jobs=-1
)

print("Training classifier...")
clf.fit(X_train_bal, y_train_bal)

print("Making predictions...")
y_pred = clf.predict(X_test_with_graph)
y_pred_proba = clf.predict_proba(X_test_with_graph)

# Results
print("\n" + "="*50)
print("CLASSIFICATION REPORT")
print("="*50)
print(classification_report(y_test, y_pred))

# Feature importance analysis
feature_names = (
    [f'num_{i}' for i in range(len(numeric_cols))] +
    [f'cat_{i}' for i in range(X_train_dense.shape[1] - len(numeric_cols))] +
    list(graph_feats_train.columns)
)

importances = clf.feature_importances_
graph_feature_importance = importances[-len(graph_feats_train.columns):].sum()
original_feature_importance = importances[:-len(graph_feats_train.columns)].sum()

print(f"\nFeature Importance Analysis:")
print(f"Original features: {original_feature_importance:.3f}")
print(f"Graph features: {graph_feature_importance:.3f}")
print(f"Graph features contribution: {graph_feature_importance/importances.sum()*100:.1f}%")

# Detailed analysis of class-specific graph features
class_aware_features = ['intra_class_deg', 'inter_class_deg', 'class_homophily']
class_feature_indices = [i for i, name in enumerate(graph_feats_train.columns) 
                        if any(cf in name for cf in class_aware_features)]

if class_feature_indices:
    class_importance = importances[-(len(graph_feats_train.columns)):][class_feature_indices].sum()
    print(f"Class-aware graph features contribution: {class_importance/importances.sum()*100:.1f}%")

# Prediction confidence analysis by class
for class_label in [0, 1]:
    class_mask = (y_test == class_label)
    if np.any(class_mask):
        class_proba = y_pred_proba[class_mask, class_label]
        print(f"\nClass {class_label} prediction confidence:")
        print(f"  Mean: {class_proba.mean():.3f}, Std: {class_proba.std():.3f}")
        print(f"  Min: {class_proba.min():.3f}, Max: {class_proba.max():.3f}")

Class distribution in training: {1.0: 1000, 0.0: 1000}
Class distribution in test: {1.0: 500, 0.0: 500}


UnboundLocalError: cannot access local variable 'minority_class' where it is not associated with a value

In [69]:
X_train_with_graph[:1,:43]

array([[-8.23162180e-02,  2.69901792e-01,  4.95739504e-01,
         1.57745916e-01,  4.01974098e+00,  7.34662351e-01,
        -1.33599968e+00,  9.31576145e-01, -1.39210974e-02,
        -1.02686520e+00, -2.25749524e-01,  1.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         1.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  1.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  6.33544515e-03,  4.75000000e-01,
         2.16374269e-01,  2.16374269e-01,  3.70000000e+01,
         1.18615911e-02,  6.80084716e-03, -3.98603348e-02,
         2.68416365e-02, -1.98396649e-02, -3.14359965e-03,
         2.20557792e-02,  8.39823114e-03,  2.09740907e-02,
         4.20656030e-02, -4.64396749e-05, -2.43138177e-02,
        -2.72309708e-02, -3.02165834e-02,  2.10810520e-02,
        -1.96676180e-03]])